# LEVELS Benchmark: DSI Data Explorer Agent

This notebook benchmarks the DSI Data Explorer agent across three task-complexity levels and five data-management categories.

## Complexity Levels

| Level | Name | Description | Scoring Method |
|-------|------|-------------|---------|
| **1** | Atomic | Single DSI/API call from natural language | Binary pass/fail: correct tool + task completion |
| **2** | Workflow | Multi-step pipelines (query → filter → summarize, ingest → validate, etc.) | Output correctness + method quality (LLM judge) |
| **3** | Discovery | Hypothesis-level: state a relationship as structured hypothesis (DiscoveryBench-style) | Faceted LLM judge + expert ground truth |

## Data Management Categories

1. Search: external/domain knowledge and literature
2. Data Discovery: locating datasets, files, or records in DSI repositories
3. Data Preparation: cleaning, validating, standardizing scientific data
4. Metadata Generation: structured documentation and provenance
5. Analysis & Summarization: interpreting, aggregating, synthesizing data

These are the DSI tools:

dsi_tools = [

                query_dsi_tool, 

                python_repl_tool, 

                load_dsi_tool, 

                download_file_tool, 

                arxiv_search_tool,

                wikipedia_search_tool,

                web_search_tool,

                send_email_tool,

                upload_paper_tool


## Setup

In [1]:
# Install dependencies into the active Jupyter kernel (run once)
%pip install -q -r requirements.txt
# --no-build-isolation: reuse the setuptools already in this kernel instead of pip
# fetching a fresh copy from PyPI for the build environment (avoids network/SSL
# failures on machines with restricted/proxied access to pypi.org).
%pip install -q --no-build-isolation -e ../..

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load portal API key from portal_api_key.txt (see portal_api_key.txt.example)
import os
from pathlib import Path

_key_path = Path("portal_api_key.txt")
if not _key_path.is_file():
    raise FileNotFoundError(
        "Missing portal_api_key.txt — copy portal_api_key.txt.example and add your key."
    )
os.environ.setdefault("API_KEY", _key_path.read_text().strip())
os.environ["API_BASE"] = "https://aiportal-api.aws.lanl.gov"

In [3]:
import threading
import contextlib
import io
import json
import re
import subprocess
import sys
import textwrap
import time
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import pandas as pd
from IPython.display import HTML, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

def find_ai_search_dir() -> Path:
    """Locate the tools/ai_search directory from cwd or any parent.

    Walks up the directory tree looking for dsi_explorer_agent.py so the
    notebook still works after the agent changes the process working directory.
    """
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "dsi_explorer_agent.py").is_file():
            return directory.resolve()
        nested = directory / "tools" / "ai_search" / "dsi_explorer_agent.py"
        if nested.is_file():
            return nested.parent.resolve()
    raise RuntimeError("Could not find tools/ai_search — open this notebook from the dsi repo")

AI_SEARCH_DIR = find_ai_search_dir()
os.chdir(AI_SEARCH_DIR)

from dsi_explorer_agent import DSIExplorer, extract_message_texts
from tool_utils import (
    reset_dsi_call_log, get_dsi_call_log,
    reset_timing_log, get_timing_log, record_timing, timed,
    summarize_timing, format_timing_summary, wall_clock_total,
    _is_nested_phase, TimingCallbackHandler,
)

print(f"Working directory: {AI_SEARCH_DIR}")


Working directory: /Users/hhn/lanl/dsi/tools/ai_search


In [4]:
# --- Configuration ---
AI_PORTAL_BASE_URL = os.environ.get("API_BASE", "https://aiportal-api.aws.lanl.gov")

POSSIBLE_MODELS = [
    "aws-gov.gpt-5.4",
    "anthropic.claude-sonnet-4-5-20250929-v1:0",
    # "amazon.nova-pro-v1:0",
    # "meta.llama3-70b-instruct-v1:0",
    # "meta.llama3-8b-instruct-v1:0",
    # "mistral7b",
    # "llama3bv32instr",
    # "phi4",
    # "gpt-oss-120b",
    # "e5mistral7b",
    # "NVIDIA-Nemotron-3-Super-120B-A12B-FP8",
    # "gemma-4-31B-it",
]

LLM_MODEL = "aws-gov.gpt-5.4"   # default single model, used by make_explorer()/debug_task() when no llm is passed
JUDGE_MODELS = list(POSSIBLE_MODELS)  # ensemble of judges: every judged score is the AVERAGE across these models,
                                        # so a model under test never sits alone judging its own (or a rival's) transcripts
MODELS_TO_RUN = list(POSSIBLE_MODELS)  # subset this to run the multi-model sweep against fewer models
TASK_TIMEOUT_SEC = 180           # max wall-clock seconds per agent turn before it's aborted as a timeout
USE_MCP = False                 # set True if levels_mcp_server.py is running on :8000
MCP_URL = "http://127.0.0.1:8000/mcp"
MASTER_DB = "data/oceans_11/ocean_11_datasets.db"

# NIF_DB = "data/oceans_11/nif.db"
# POISSON_DB = "data/oceans_11/poisson.db"

RESULTS_DIR = AI_SEARCH_DIR / "levels_benchmark_results"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def make_llm(model_name: str, temperature: float = 0.1) -> ChatOpenAI:
    """Create a ChatOpenAI client pointed at the LANL AI portal."""
    return ChatOpenAI(
        model=model_name,
        api_key=os.environ["API_KEY"],
        base_url=AI_PORTAL_BASE_URL,
        temperature=temperature,
    )

llm_model = make_llm(LLM_MODEL)
judge_llms = [make_llm(m, temperature=0.0) for m in JUDGE_MODELS]
judge_llm = judge_llms[0]  # back-compat alias for any direct judge_llm.invoke() usage
print(f"Using model: {LLM_MODEL} @ {AI_PORTAL_BASE_URL}")
print(f"Judge ensemble: {JUDGE_MODELS}")


Using model: aws-gov.gpt-5.4 @ https://aiportal-api.aws.lanl.gov
Judge ensemble: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0']


In [5]:
def make_explorer(llm: Optional[ChatOpenAI] = None) -> DSIExplorer:
    """Create a fresh DSIExplorer with a new workspace, thread, and master DB loaded.

    Resets cwd to AI_SEARCH_DIR first to avoid nested workspace folders on re-runs.
    Passes MCP settings when USE_MCP is True. Defaults to the global llm_model
    (LLM_MODEL) when no llm is given, so existing single-model calls keep working;
    pass a per-model client (e.g. make_llm(model_name)) to benchmark another model.
    """
    os.chdir(AI_SEARCH_DIR)  # avoid nested workspace folders on re-runs
    return DSIExplorer(
        llm=llm if llm is not None else llm_model,
        db_index_name=MASTER_DB,
        output_mode="jupyter",
        **({"use_mcp": True, "mcp_url": MCP_URL} if USE_MCP else {}),
    )


## Benchmark Harness

In [6]:
# Ground-truth constants for Level-1 quantitative checks
NIF_DATASET_ID = "3D FLASH Computation of National Ignition Facility Shot"


def canonicalize_db_path(path: str) -> str:
    """Normalize a DB path for exact Level-1 path matching."""
    s = Path(str(path)).as_posix()
    if "oceans_11/" in s:
        return s[s.index("oceans_11/") :]
    return Path(s).name


def _resolve_path_arg(call_args: Dict[str, Any]) -> str:
    """Resolve a load_dsi_tool 'path' arg the same way load_dsi() resolves it internally.

    Both models converged on passing the master/secondary DB's bare filename as `path`
    plus the folder as a separate `master_db_folder` kwarg (load_dsi_tool's own documented
    pattern for resolving relative paths against the master DB's folder) rather than a
    single joined string. Without this, canonicalize_db_path("nif.db") != canonicalize_db_path
    ("oceans_11/nif.db") even though the model loaded the exact right file.
    """
    path = str(call_args.get("path", ""))
    folder = call_args.get("master_db_folder")
    if folder and not Path(path).is_absolute():
        return str(Path(str(folder)) / path)
    return path


def _safe_model_label(model_name: str) -> str:
    """Sanitize a model name (which may contain ':', '.', '/') into a filename-safe label."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)


@dataclass
class BenchmarkTask:
    """Definition of a single benchmark task (query, rubric, and scoring hints)."""
    task_id: str
    level: int
    category: str
    query: str
    # Level 1 checks
    expected_tools: List[str] = field(default_factory=list)
    expected_args: Dict[str, Any] = field(default_factory=dict)
    expected_path: Optional[str] = None  # canonicalize and compare to tool arg "path"
    expected_count: Optional[int] = None  # must appear in tool result or response
    response_keywords: List[str] = field(default_factory=list)
    forbidden_keywords: List[str] = field(default_factory=list)
    reference_path: List[str] = field(default_factory=list)  # most-efficient human DSI CLI commands, for LLM-judged efficiency scoring
    reference_time_sec: Optional[float] = None  # human wall-clock seconds to execute reference_path manually; RQ1 productivity baseline. Must be measured by a human, not estimated -- left None until that data-collection pass happens.
    setup_query: Optional[str] = None
    timeout_sec: Optional[float] = None  # per-task override; falls back to TASK_TIMEOUT_SEC when None
    steps: List[str] = field(default_factory=list)  # ordered natural-language turns for multi-step Level 2/3 workflows, sent to the SAME explorer so later turns can rely on earlier conversational state. When empty, `query` is sent as a single turn (unchanged Level-1 behavior).
    # Level 2/3 rubric
    success_criteria: str = ""  # free-text rubric judged by score_level2's LLM judge (process-quality/method score)
    ground_truth_hypothesis: Optional[Dict[str, str]] = None  # DiscoveryBench-style facets for score_level3: {"context": ..., "variables": ..., "relationship": ...}
    # Calibration / uncertainty (RQ4): tasks referencing missing data, nonexistent resources, or
    # false/ambiguous premises, where correctly abstaining/asking for clarification IS the pass condition.
    expects_abstention: bool = False
    abstention_keywords: List[str] = field(default_factory=list)  # task-specific hedge phrases; falls back to DEFAULT_ABSTENTION_PHRASES when empty
    requires_network: bool = False
    skip: bool = False
    # Prompt-robustness testing (RQ: does prompt phrasing/quality affect success, holding the
    # underlying task/ground-truth fixed?). Hand-authored per task, not LLM-generated, so every
    # variant is verified to still unambiguously convey the facts the existing scoring checks
    # (expected_path/expected_args/response_keywords/etc.) require -- only register/tone/structure
    # varies, not substance. Value is either a str (single-turn `query` replacement) or a
    # List[str] (replaces multi-turn `steps`, for tasks like L2_cross_db_record_compare).
    prompt_variants: Dict[str, Any] = field(default_factory=dict)
    prompt_style: str = "standard"  # set on an EXPANDED per-style copy by expand_prompt_variants(); "standard" = this task's original query/steps wording
    base_task_id: str = ""  # set on an expanded copy to the pre-expansion task_id, so variants of the same task can be grouped back together; empty on un-expanded tasks


@dataclass
class TaskResult:
    """Outcome of running and scoring one benchmark task."""
    task_id: str
    level: int
    category: str
    query: str
    response: str
    tool_calls: List[Dict[str, Any]]
    elapsed_sec: float
    tokens: int
    passed: Optional[bool] = None
    score: Optional[float] = None
    judge_scores: Optional[Dict[str, Any]] = None
    notes: str = ""
    tool_results: List[str] = field(default_factory=list)
    dsi_calls: List[Dict[str, Any]] = field(default_factory=list)
    timing_log: List[Dict[str, Any]] = field(default_factory=list)  # per-phase wall-clock breakdown (agent run + scoring); see summarize_timing()/format_timing_summary()
    model: str = ""  # which LLM produced this result; set by run_benchmark/run_benchmark_all_models
    checkpoint_events: List[Dict[str, Any]] = field(default_factory=list)  # RQ3/RQ7: high-stakes tool calls + hedge language detected post-hoc (metrics only, does not pause execution)
    failure_category: Optional[str] = None  # RQ5: MODEL_REASONING / TOOL_FRAMEWORK / TASK_SPECIFICATION / PROMPT_GAP / ENVIRONMENT_DATA, set by classify_failure() for failed tasks
    failure_rationale: str = ""
    check_results: Optional[Dict[str, bool]] = None  # per-check pass/fail breakdown (e.g. Level 1's tool_ok/args_ok/.../nonempty_ok), set by score_level1() so partial credit is auditable


def expand_prompt_variants(tasks: List[BenchmarkTask]) -> List[BenchmarkTask]:
    """Expand each task's hand-authored `prompt_variants` into separate BenchmarkTask copies,
    one per style (plus "standard" = the task's original query/steps wording), so the SAME
    ground truth/scoring rubric gets tested against multiple phrasings of the same underlying
    ask. Tasks with no `prompt_variants` pass through unchanged as a single "standard" copy.

    Each expanded copy gets a unique task_id (original + "__" + style, unless style is
    "standard"), `prompt_style` set to the style name, and `base_task_id` set to the
    original task_id so variants of the same task can be grouped back together in analysis.

    A variant value that's a List[str] replaces `steps` (for multi-turn tasks like
    L2_cross_db_record_compare); a plain str replaces `query` (everything else).
    """
    expanded: List[BenchmarkTask] = []
    for t in tasks:
        standard = list(t.steps) if t.steps else t.query
        styles: Dict[str, Any] = {"standard": standard, **t.prompt_variants}
        for style, variant in styles.items():
            new_id = t.task_id if style == "standard" else f"{t.task_id}__{style}"
            if isinstance(variant, list):
                expanded.append(replace(
                    t, task_id=new_id, steps=variant, query=(variant[-1] if variant else t.query),
                    prompt_variants={}, prompt_style=style, base_task_id=t.task_id,
                ))
            else:
                expanded.append(replace(
                    t, task_id=new_id, query=variant, steps=[],
                    prompt_variants={}, prompt_style=style, base_task_id=t.task_id,
                ))
    return expanded


def extract_tool_calls(result: dict) -> List[Dict[str, Any]]:
    """Collect tool names and args from all AIMessage tool_calls in an agent result."""
    calls = []
    for msg in result.get("messages", []):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                calls.append({"name": tc.get("name"), "args": tc.get("args", {})})
    return calls


def extract_tool_results(result: dict) -> List[str]:
    """Collect ToolMessage contents as strings for quantitative Level-1 checks."""
    out = []
    for msg in result.get("messages", []):
        if isinstance(msg, ToolMessage):
            content = msg.content
            if isinstance(content, (list, dict)):
                out.append(json.dumps(content, default=str))
            else:
                out.append(str(content))
    return out


def format_dsi_call_log(dsi_calls: List[Dict[str, Any]]) -> str:
    """Pretty-print recorded DSI Python API calls (method, args, kwargs, timing)."""
    if not dsi_calls:
        return "(no DSI API calls)"
    lines = []
    for i, call in enumerate(dsi_calls, 1):
        lines.append(
            f"[{i}] DSI.{call['method']}("
            f"{', '.join(call.get('args', []))}"
            f"{', ' if call.get('args') and call.get('kwargs') else ''}"
            f"{', '.join(f'{k}={v}' for k, v in call.get('kwargs', {}).items())})"
            f"  [{call.get('elapsed_sec', 0):.3f}s]"
            + (f"  ERROR: {call['error']}" if call.get("error") else "")
        )
    return "\n".join(lines)


class TaskTimeoutError(TimeoutError):
    """Raised when a single agent invocation exceeds its allotted timeout."""


def invoke_agent(explorer: DSIExplorer, query: str, timeout: Optional[float] = None, scope: str = "") -> dict:
    """Send one user query through the agent graph and return the raw result dict.

    Uses async invoke when MCP is enabled; otherwise sync invoke. Adds _elapsed
    seconds to the result for timing metrics.

    `scope` (e.g. "setup_query", "step:1") tags every llm_call/tool:* timing entry recorded
    during this invocation with which top-level phase it belongs to, so format_timing_summary()
    can attribute nested calls to the right step instead of merging every step of a multi-turn
    task into one bucket. Pass the same label used for the enclosing timed(...) block in run_task.

    The (blocking) invoke call runs on a daemon thread; if it doesn't finish
    within `timeout` seconds (defaults to TASK_TIMEOUT_SEC), TaskTimeoutError is
    raised so a stuck/slow model can't hang the whole benchmark sweep. Python
    cannot forcibly kill a running thread, so the abandoned call keeps running
    in the background — but because the thread is a daemon, it won't block the
    notebook kernel process from exiting/restarting later, and this function
    itself returns control immediately once `timeout` elapses either way.

    Any stdout/stderr the worker thread writes is captured into a buffer
    instead of going to the real stdout from a background thread. ipykernel
    routes stream output to the currently-executing cell using a parent-header
    that gets confused by concurrent writes from multiple threads; left
    unguarded this silently drops/misattributes *later* print() calls made on
    the main thread (e.g. the "Tools called"/"DSI API calls" lines), which is
    why they tend to stop showing up starting with the second model in a
    multi-model sweep. The buffer is replayed on the main thread right after
    `join()` returns, so only the main thread ever touches the real stdout.
    """
    start = time.time()
    msg = explorer.craft_message(query)
    timeout = TASK_TIMEOUT_SEC if timeout is None else timeout

    # TimingCallbackHandler times every LLM call ("llm_call") and tool call ("tool:<name>")
    # triggered inside this one graph invocation, recording into the shared timing log so
    # run_task/score_task can later report "where did the time go" for the whole task.
    timing_cb = TimingCallbackHandler(scope=scope)
    graph_config = {"configurable": {"thread_id": explorer.thread_id}, "callbacks": [timing_cb]}

    box: Dict[str, Any] = {}

    def _invoke():
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                if explorer.use_mcp:
                    from dsi_explorer_agent import run_async
                    box["result"] = run_async(explorer.app.ainvoke(msg, config=graph_config))
                else:
                    box["result"] = explorer.app.invoke(msg, config=graph_config)
        except Exception as e:
            box["error"] = e
        finally:
            box["captured_output"] = buf.getvalue()

    worker = threading.Thread(target=_invoke, daemon=True)
    worker.start()
    worker.join(timeout=timeout)
    if worker.is_alive():
        raise TaskTimeoutError(f"Agent call exceeded {timeout}s timeout for query: {query[:80]!r}")

    captured = box.get("captured_output", "")
    if captured:
        sys.stdout.write(captured)
        sys.stdout.flush()

    if "error" in box:
        raise box["error"]

    result = box["result"]
    result["_elapsed"] = time.time() - start
    return result


def run_task(explorer: DSIExplorer, task: BenchmarkTask, verbose: bool = True) -> TaskResult:
    """Execute one benchmark task and capture response, tool calls, and timing.

    Runs setup_query first when present (e.g. load a DB before the main query).
    When task.steps is non-empty (Level 2/3 multi-turn workflows), each step is sent in
    order to the SAME explorer, so later steps can rely on earlier conversational state
    (e.g. "which of the two datasets you just examined..."); tool_calls/tool_results/tokens/
    elapsed_sec are accumulated across all steps and the LAST step's reply becomes
    result.response. When task.steps is empty, behaves exactly as before (single-turn
    task.query) -- this is unchanged for every existing Level 1 task.
    Skipped tasks return immediately with notes='skipped'.
    """
    if task.skip:
        return TaskResult(task.task_id, task.level, task.category, task.query,
                          "", [], 0.0, 0, passed=None, notes="skipped")

    reset_dsi_call_log()
    reset_timing_log()  # cleared here; score_task() reads the accumulated log at the end so it covers agent run + scoring

    if task.setup_query:
        with timed("setup_query"):
            invoke_agent(explorer, task.setup_query, timeout=task.timeout_sec, scope="setup_query")

    if verbose:
        display(Markdown(f"### `{task.task_id}` — Level {task.level} / {task.category}\n\n**Query:** {task.query}"))

    turns = task.steps if task.steps else [task.query]
    response = ""
    tool_calls: List[Dict[str, Any]] = []
    tool_results: List[str] = []
    tokens = 0
    elapsed = 0.0
    for i, turn_query in enumerate(turns):
        if verbose and task.steps:
            display(Markdown(f"**Step {i + 1}/{len(turns)}:** {turn_query}"))
        step_scope = f"step:{i + 1}"
        with timed(step_scope, query=turn_query[:60]):
            step_result = invoke_agent(explorer, turn_query, timeout=task.timeout_sec, scope=step_scope)
        response = (step_result.get("response") or "").strip()
        tool_calls.extend(extract_tool_calls(step_result))
        tool_results.extend(extract_tool_results(step_result))
        tokens += step_result.get("metadata", {}).get("token_usage", {}).get("total_tokens", 0)
        elapsed += step_result["_elapsed"]
        if verbose:
            display(Markdown(response))

    dsi_calls = get_dsi_call_log()

    if verbose:
        print(f"Tools called: {[t['name'] for t in tool_calls]} | {elapsed:.1f}s | {tokens} tokens")
        print(f"DSI API calls: {[c['method'] for c in dsi_calls]}")

    return TaskResult(
        task_id=task.task_id,
        level=task.level,
        category=task.category,
        query=task.query,
        response=response,
        tool_calls=tool_calls,
        elapsed_sec=elapsed,
        tokens=tokens,
        tool_results=tool_results,
        dsi_calls=dsi_calls,
    )


def score_level1(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 1 (atomic) task with 7 independent binary checks, partial credit.

    Checks: expected tool, expected args (exact), canonicalized path, quantitative
    count in tool results/response, keywords / forbidden keywords, nonempty response.
    Each check defaults to True (vacuously satisfied) when the task doesn't define the
    corresponding field, exactly as before.

    result.check_results stores every check's individual True/False so the breakdown is
    auditable (and displayable per-task, e.g. in l1_df). result.score is now the FRACTION
    of the 7 checks that passed (accumulated partial credit, 0.0-1.0) rather than a strict
    0.0/1.0 binary -- e.g. a task that nails 6 of 7 checks now scores ~0.86 instead of 0.0.
    result.passed remains the strict all-checks-pass flag (unchanged), since pass_rate
    aggregations elsewhere in the notebook rely on it meaning "fully correct".
    """
    tool_names = [t["name"] for t in result.tool_calls]
    matching_calls = [t for t in result.tool_calls if t["name"] in (task.expected_tools or tool_names)]
    notes = []

    tool_ok = True
    if task.expected_tools:
        tool_ok = any(t in tool_names for t in task.expected_tools)
        if not tool_ok:
            notes.append(f"Expected one of {task.expected_tools}, got {tool_names}")

    args_ok = True
    if task.expected_args and tool_ok:
        args_ok = False

        def _arg_match(call_args: Dict[str, Any], key: str, want: Any) -> bool:
            if key == "path":
                return canonicalize_db_path(_resolve_path_arg(call_args)) == canonicalize_db_path(str(want))
            return str(call_args.get(key)) == str(want)

        for call in matching_calls:
            if all(_arg_match(call["args"], k, v) for k, v in task.expected_args.items()):
                args_ok = True
                break
        if not args_ok:
            notes.append(
                f"Expected args {task.expected_args}, got "
                f"{[c['args'] for c in matching_calls]}"
            )

    path_ok = True
    if task.expected_path is not None and tool_ok:
        path_ok = False
        want = canonicalize_db_path(task.expected_path)
        for call in matching_calls:
            got = _resolve_path_arg(call["args"])
            if got and canonicalize_db_path(got) == want:
                path_ok = True
                break
        if not path_ok:
            notes.append(
                f"Expected canonical path {want}, got "
                f"{[c['args'].get('path') for c in matching_calls]}"
            )

    quant_ok = True
    if task.expected_count is not None:
        blob = " ".join(result.tool_results + [result.response])
        # Prefer structured count keys, then bare integer token.
        patterns = [
            rf'["\']?(?:count|missing_count)["\']?\s*[:=]\s*{task.expected_count}\b',
            rf"\b{task.expected_count}\b",
        ]
        quant_ok = any(re.search(p, blob) for p in patterns)
        if not quant_ok:
            notes.append(f"Expected count {task.expected_count} not found in tool results/response")

    keyword_ok = True
    response_lower = result.response.lower()
    tools_blob = " ".join(result.tool_results).lower() + " " + response_lower
    if task.response_keywords:
        keyword_ok = any(kw.lower() in tools_blob for kw in task.response_keywords)
        if not keyword_ok:
            notes.append(f"Response missing keywords: {task.response_keywords}")

    forbidden_ok = True
    if task.forbidden_keywords:
        hits = [kw for kw in task.forbidden_keywords if kw.lower() in tools_blob]
        if hits:
            forbidden_ok = False
            notes.append(f"Forbidden distractors present: {hits}")

    # Atomic tools may return short structured answers; allow short responses when tools ran.
    nonempty = len(result.response.strip()) > 0 or bool(result.tool_calls)
    if not nonempty:
        notes.append("Response too short or empty")

    checks = {
        "tool_ok": tool_ok,
        "args_ok": args_ok,
        "path_ok": path_ok,
        "quant_ok": quant_ok,
        "keyword_ok": keyword_ok,
        "forbidden_ok": forbidden_ok,
        "nonempty_ok": nonempty,
    }
    result.check_results = checks
    result.passed = all(checks.values())
    result.score = round(sum(checks.values()) / len(checks), 3)
    result.notes = "; ".join(notes)
    return result


def llm_judge(prompt: str, llm: Optional[ChatOpenAI] = None) -> dict:
    """Ask a judge LLM to score a task and parse JSON from its response.

    Defaults to the first configured judge model (judge_llm) when `llm` is omitted; pass an
    explicit `llm` (e.g. from judge_llms) to target a specific ensemble member.
    Falls back to {"raw": text} if the model returns non-JSON output.
    """
    resp = (llm or judge_llm).invoke(prompt)
    text = resp.content.strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"raw": text}


def llm_judge_ensemble(prompt: str) -> List[Dict[str, Any]]:
    """Run the SAME judge prompt against every model in JUDGE_MODELS and return every verdict,
    each tagged with which model produced it ("_judge_model").

    This is what lets every judged score in the harness be an AVERAGE across judge models
    rather than a single model's opinion -- important because JUDGE_MODELS overlaps with
    MODELS_TO_RUN, so without an ensemble a model would sometimes be judging its own (or a
    rival's) transcripts alone. Callers average whichever numeric field(s) they care about
    via _avg_ensemble_field(); for categorical verdicts (see classify_failure), callers check
    for ensemble agreement instead.
    """
    verdicts = []
    for model_name, llm in zip(JUDGE_MODELS, judge_llms):
        verdict = llm_judge(prompt, llm=llm)
        verdict["_judge_model"] = model_name
        verdicts.append(verdict)
    return verdicts


def _avg_ensemble_field(verdicts: List[Dict[str, Any]], key: str) -> Optional[float]:
    """Mean of a numeric field across ensemble verdicts, ignoring judges that omitted it
    (e.g. returned malformed JSON). Returns None if no judge produced a usable value.
    """
    vals = [float(v[key]) for v in verdicts if isinstance(v.get(key), (int, float))]
    return round(sum(vals) / len(vals), 3) if vals else None


def _format_ensemble_reasoning(verdicts: List[Dict[str, Any]], reasoning_key: str = "reasoning") -> str:
    """Concatenate each judge's rationale, prefixed by which model said it, for auditability."""
    return " | ".join(
        f"[{v.get('_judge_model')}] {v.get(reasoning_key, v.get('raw', ''))}" for v in verdicts
    )


def score_efficiency(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """LLM-judge how efficiently the agent solved the task vs. the human reference CLI path.

    Uses task.reference_path (the most-efficient sequence of DSI CLI commands, written by a
    human) as the gold standard, and compares it against the agent's actual tool_calls and
    dsi_calls. The verdict is stored in result.judge_scores so it auto-flattens into the
    results dataframe as judge_efficiency_score / judge_efficiency_reasoning, alongside (and
    independent of) the binary correctness score from score_level1. Tasks with no
    reference_path are skipped (left unscored for efficiency).
    """
    if not task.reference_path:
        return result

    agent_tool_calls = [f"{c['name']}({c['args']})" for c in result.tool_calls]
    agent_dsi_calls = [
        f"{c['method']}({', '.join(c.get('args', []))})" for c in result.dsi_calls
    ]

    prompt = f"""You are judging the EFFICIENCY of an AI agent's path for a data-management task,
against the most efficient path a human expert would take at the DSI command line.

Task: {task.query}

Reference path (ground truth, most efficient -- one command per line):
{chr(10).join(task.reference_path)}

Agent's actual tool calls (in order):
{json.dumps(agent_tool_calls, indent=2)}

Agent's actual underlying DSI API calls (in order):
{json.dumps(agent_dsi_calls, indent=2)}

Score the agent's efficiency from 0.0 (far less efficient than the reference -- redundant
queries, unnecessary exploration, retries, or extra DSI calls beyond what the reference
path needed) to 1.0 (matched the reference path's number of essential steps with no waste).
A correct final answer reached via a wasteful path should still score low here.

Respond with ONLY JSON: {{"efficiency_score": <float 0.0-1.0>, "reasoning": "<one sentence>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    result.judge_scores = result.judge_scores or {}
    result.judge_scores["efficiency_score"] = _avg_ensemble_field(verdicts, "efficiency_score")
    result.judge_scores["efficiency_score_by_judge"] = {
        v.get("_judge_model"): v.get("efficiency_score") for v in verdicts
    }
    result.judge_scores["efficiency_reasoning"] = _format_ensemble_reasoning(verdicts)
    return result


def score_level2(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 2 (workflow) task: tool coverage + quantitative/keyword checks on the
    final response, plus an LLM-judged process-quality score against task.success_criteria.

    Unlike score_level1 (which only requires ANY ONE of expected_tools, since a Level 1
    query has one right tool), a Level 2 workflow must exercise EVERY tool in
    expected_tools at least once -- e.g. a search-then-download-then-summarize task isn't
    "correct" if the agent skipped the download step. Similarly response_keywords are
    ALL-required here (a workflow answer typically needs to report several distinct facts),
    not any-of like Level 1's alternate-phrasing check.

    result.score blends binary correctness (0.5 weight) with the continuous LLM-judged
    process_quality_score (0.5 weight) when success_criteria is set, so a correct-but-
    wasteful or correct-but-poorly-ordered workflow scores lower than a clean one, mirroring
    how score_efficiency already treats Level 1. result.passed reflects correctness only.
    """
    tool_names = [t["name"] for t in result.tool_calls]
    notes = []

    tools_ok = True
    if task.expected_tools:
        missing = [t for t in task.expected_tools if t not in tool_names]
        tools_ok = not missing
        if missing:
            notes.append(f"Missing required tools for this workflow: {missing} (called: {tool_names})")

    blob = " ".join(result.tool_results + [result.response])
    blob_lower = blob.lower()

    quant_ok = True
    if task.expected_count is not None:
        patterns = [
            rf'["\']?(?:count|missing_count)["\']?\s*[:=]\s*{task.expected_count}\b',
            rf"\b{task.expected_count}\b",
        ]
        quant_ok = any(re.search(p, blob) for p in patterns)
        if not quant_ok:
            notes.append(f"Expected count {task.expected_count} not found in tool results/response")

    keyword_ok = True
    if task.response_keywords:
        keyword_ok = all(kw.lower() in blob_lower for kw in task.response_keywords)
        if not keyword_ok:
            missing_kw = [kw for kw in task.response_keywords if kw.lower() not in blob_lower]
            notes.append(f"Response missing required facts/keywords: {missing_kw}")

    forbidden_ok = True
    if task.forbidden_keywords:
        hits = [kw for kw in task.forbidden_keywords if kw.lower() in blob_lower]
        if hits:
            forbidden_ok = False
            notes.append(f"Forbidden distractors present: {hits}")

    nonempty = len(result.response.strip()) > 0
    if not nonempty:
        notes.append("Response too short or empty")

    correctness_ok = all([tools_ok, quant_ok, keyword_ok, forbidden_ok, nonempty])

    process_score = None
    if task.success_criteria:
        prompt = f"""You are judging the METHOD QUALITY of an AI agent's multi-step workflow for a
data-management task, against a rubric describing what a good approach looks like.

Task: {task.query}

Rubric (what a high-quality workflow should do): {task.success_criteria}

Agent's tool calls (in order): {json.dumps([f"{c['name']}({c['args']})" for c in result.tool_calls], indent=2)}

Agent's final response: {result.response}

Score process quality from 0.0 (ignored the rubric, wrong/haphazard approach) to 1.0 (followed
a sound, efficient multi-step approach matching the rubric's intent).

Respond with ONLY JSON: {{"process_quality_score": <float 0.0-1.0>, "reasoning": "<one sentence>"}}
"""
        verdicts = llm_judge_ensemble(prompt)
        process_score = _avg_ensemble_field(verdicts, "process_quality_score")
        result.judge_scores = result.judge_scores or {}
        result.judge_scores["process_quality_score"] = process_score
        result.judge_scores["process_quality_score_by_judge"] = {
            v.get("_judge_model"): v.get("process_quality_score") for v in verdicts
        }
        result.judge_scores["process_quality_reasoning"] = _format_ensemble_reasoning(verdicts)

    result.passed = correctness_ok
    correctness_component = 1.0 if correctness_ok else 0.0
    if isinstance(process_score, (int, float)):
        result.score = round(0.5 * correctness_component + 0.5 * float(process_score), 3)
    else:
        result.score = correctness_component
    result.notes = "; ".join(notes)
    return result


def score_level3(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 3 (discovery/hypothesis) task with a DiscoveryBench-style faceted judge.

    Compares the agent's final response against task.ground_truth_hypothesis facets
    (context, variables, relationship) written by a human who examined the same data, each
    scored 0.0-1.0 by the judge LLM. result.score is the mean facet score; result.passed uses
    a 0.6 threshold rather than Level 1/2's stricter binary checks, since partial credit is
    the norm for open-ended discovery (the agent may find the right relationship but describe
    the wrong variables, etc.) -- see judge_scores/notes for which facet(s) were missed.
    """
    gt = task.ground_truth_hypothesis or {}
    if not gt:
        result.notes = "no ground_truth_hypothesis defined on this task; skipping level-3 scoring"
        result.score = None
        result.passed = None
        return result

    prompt = f"""You are judging a scientific HYPOTHESIS an AI agent proposed after analyzing a DSI
scientific dataset, against a ground-truth hypothesis written by a human expert who examined the
same data (DiscoveryBench-style faceted scoring). Score each facet from 0.0 (not addressed or
contradicted) to 1.0 (fully matches in substance, wording may differ).

Task: {task.query}

Ground-truth hypothesis facets:
- context: {gt.get('context', '')}
- variables: {gt.get('variables', '')}
- relationship: {gt.get('relationship', '')}

Agent's final response:
{result.response}

Respond with ONLY JSON: {{"context_score": <0.0-1.0>, "variables_score": <0.0-1.0>,
"relationship_score": <0.0-1.0>, "reasoning": "<1-2 sentences noting any facet missed or contradicted>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    context_score = _avg_ensemble_field(verdicts, "context_score")
    variables_score = _avg_ensemble_field(verdicts, "variables_score")
    relationship_score = _avg_ensemble_field(verdicts, "relationship_score")
    facet_scores = [s for s in (context_score, variables_score, relationship_score) if isinstance(s, (int, float))]
    hypothesis_reasoning = _format_ensemble_reasoning(verdicts)

    result.judge_scores = result.judge_scores or {}
    result.judge_scores.update({
        "context_score": context_score,
        "variables_score": variables_score,
        "relationship_score": relationship_score,
        "hypothesis_reasoning": hypothesis_reasoning,
    })
    result.score = round(sum(facet_scores) / len(facet_scores), 3) if facet_scores else 0.0
    result.passed = result.score >= 0.6
    result.notes = hypothesis_reasoning
    return result


# RQ4 (does the agent know what it doesn't know?): fallback hedge/abstention phrases used by
# score_calibration() when a task doesn't define its own abstention_keywords.
DEFAULT_ABSTENTION_PHRASES = [
    "not found", "no such", "does not exist", "doesn't exist", "cannot find", "can't find",
    "could not find", "couldn't find", "unable to determine", "unable to locate",
    "no information", "not available", "not specified", "no matching", "no records",
    "none found", "no datasets", "not documented", "could you clarify", "could you specify",
    "which dataset", "can you clarify", "not sure", "i don't know", "i do not know",
    "no such column", "no such field", "unclear", "does not specify", "not provided",
]


def score_calibration(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a calibration ("trap") task (RQ4): the task references missing data, a
    nonexistent resource, or a false/ambiguous premise, and the CORRECT behavior is to
    abstain or ask for clarification rather than confidently fabricate an answer.

    Passes (score=1.0) if the response contains abstention/clarification language
    (task.abstention_keywords, falling back to DEFAULT_ABSTENTION_PHRASES); otherwise fails
    (score=0.0) and result.notes is tagged "HALLUCINATION" so it's easy to filter for in
    analysis. This is a cheap keyword heuristic, not an LLM judge -- it will have false
    negatives (an agent that hedges in an unanticipated phrasing) but very few false
    positives, which is the safer failure mode for measuring overconfidence.
    """
    phrases = [p.lower() for p in (task.abstention_keywords or DEFAULT_ABSTENTION_PHRASES)]
    response_lower = result.response.lower()
    abstained = any(p in response_lower for p in phrases)

    result.passed = abstained
    result.score = 1.0 if abstained else 0.0
    result.notes = (
        "correctly abstained / flagged uncertainty" if abstained
        else "HALLUCINATION: answered confidently despite missing, nonexistent, or ambiguous data"
    )
    return result


# RQ3/RQ7 (autonomy & human-guidance): tools whose effects are hard to undo (writes data to
# disk, sends external email, uploads to a third-party API) -- a real human-in-the-loop system
# would plausibly gate these behind an approval step.
HIGH_STAKES_TOOLS = {"download_file_tool", "send_email_tool", "upload_paper_tool"}
# Hedging language in the agent's own final response -- a signal the agent itself was unsure.
HEDGE_PHRASES = [
    "i think", "probably", "not certain", "not sure", "may be", "might be", "possibly",
    "i believe", "seems like", "appears to be", "could be wrong", "unclear", "i'm not 100%",
]


def detect_checkpoint_opportunities(result: TaskResult) -> TaskResult:
    """Post-hoc, METRICS-ONLY detector for where a human-in-the-loop checkpoint would have
    fired had one existed (RQ3/RQ7). This does not pause, gate, or alter execution in any
    way -- it only annotates result.checkpoint_events so later analysis can ask "how often
    would a human have been consulted, and did that correlate with pass/fail?" as an
    observational first pass. See BENCHMARK_METHODOLOGY.md for the proposed follow-on study
    that actually runs the same tasks under simulated autonomy tiers.

    Flags two kinds of events per task: (a) any call to a HIGH_STAKES_TOOLS tool (an
    irreversible/external-effect action), (b) HEDGE_PHRASES found in the final response.
    """
    events = []
    for call in result.tool_calls:
        if call.get("name") in HIGH_STAKES_TOOLS:
            events.append({"type": "high_stakes_action", "tool": call["name"], "args": call.get("args", {})})
    response_lower = result.response.lower()
    for phrase in HEDGE_PHRASES:
        if phrase in response_lower:
            events.append({"type": "hedge_language", "phrase": phrase})
    result.checkpoint_events = events
    return result


# RQ5 (when the agent fails, why?): failure-attribution taxonomy.
FAILURE_CATEGORIES = [
    "MODEL_REASONING",    # tools/data were available; the model chose poorly or misread a result
    "TOOL_FRAMEWORK",     # a tool exception, agent-framework bug, or timeout occurred
    "TASK_SPECIFICATION", # the task/rubric is ambiguous or the scorer is too strict for a reasonable answer
    "PROMPT_GAP",         # the agent's system prompt (dsi_explorer_agent.py) lacked guidance that would have helped
    "ENVIRONMENT_DATA",   # network/API/file unavailability outside the agent's control
]


def classify_failure(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """LLM-judge the most likely ROOT CAUSE of a failed task (RQ5), so failures can be
    attributed to the model, the agent framework, the task/rubric design, the system
    prompt, or the environment instead of being lumped into one pass/fail bucket.

    Only meaningful on a TaskResult that already failed (result.passed is False); left
    untouched otherwise. Feeds the judge LLM the query, the scorer's own notes (which
    already often hint at the cause, e.g. "timeout:" or "error:"), the agent's tool calls,
    and its final response, and asks for exactly one category from FAILURE_CATEGORIES plus
    a one-sentence rationale, stored on result.failure_category / result.failure_rationale.

    Unlike the numeric judges (efficiency/process-quality/hypothesis), a category label can't
    be "averaged" across the judge ensemble -- instead this uses AGREEMENT: if every judge in
    JUDGE_MODELS picks the same category, that's the result; if they disagree, the category is
    "UNKNOWN" rather than arbitrarily picking one judge's opinion, and every judge's individual
    pick + rationale is preserved in result.failure_rationale for manual review.
    """
    if result.passed:
        return result

    prompt = f"""An AI agent failed a data-management benchmark task. Classify the MOST LIKELY
root cause into exactly one of these categories:

- MODEL_REASONING: tools/data were available; the model picked the wrong approach or misread a result
- TOOL_FRAMEWORK: a tool exception, agent-framework bug, or timeout occurred
- TASK_SPECIFICATION: the task or its scoring rubric is ambiguous / overly strict for a reasonable answer
- PROMPT_GAP: the agent's system prompt lacked guidance that would have prevented this
- ENVIRONMENT_DATA: network, external API, or file unavailability outside the agent's control

Task query: {task.query}
Scorer notes: {result.notes}
Agent's tool calls: {json.dumps(result.tool_calls, default=str)[:2000]}
Agent's final response: {result.response[:1000]}

Respond with ONLY JSON: {{"failure_category": "<one of the five category names above, exactly as spelled>", "rationale": "<one sentence>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    categories = [v.get("failure_category") for v in verdicts]
    valid_categories = [c for c in categories if c in FAILURE_CATEGORIES]
    if valid_categories and len(set(valid_categories)) == 1:
        result.failure_category = valid_categories[0]
    else:
        result.failure_category = "UNKNOWN"
    result.failure_rationale = " | ".join(
        f"[{v.get('_judge_model')}] {v.get('failure_category')}: {v.get('rationale', v.get('raw', ''))}"
        for v in verdicts
    )
    return result


def score_task(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Route a TaskResult to the scorer for its complexity level, then layer on efficiency,
    checkpoint-opportunity detection, and (for failures) root-cause classification.

    Calibration ("trap") tasks are routed by task.expects_abstention regardless of level,
    since the pass condition there (abstain/ask for clarification) is orthogonal to the
    level 1/2/3 correctness rubrics.
    """
    if task.expects_abstention:
        with timed("score:calibration"):
            result = score_calibration(task, result)
    elif task.level == 1:
        with timed("score:level1"):
            result = score_level1(task, result)
    elif task.level == 2:
        with timed("score:level2"):
            result = score_level2(task, result)
    else:
        with timed("score:level3"):
            result = score_level3(task, result)
    with timed("score:efficiency"):
        result = score_efficiency(task, result)
    with timed("score:checkpoint_opportunities"):
        result = detect_checkpoint_opportunities(result)
    with timed("score:classify_failure"):
        result = classify_failure(task, result)
    # Captured last so it covers everything recorded since run_task()'s reset_timing_log():
    # agent run (setup/steps/llm_call/tool:*) plus every scoring phase above.
    result.timing_log = get_timing_log()
    return result


def run_benchmark(
    tasks: List[BenchmarkTask],
    *,
    isolated: bool = False,
    llm: Optional[ChatOpenAI] = None,
    model_label: str = "",
) -> List[TaskResult]:
    """Run and score a list of tasks, printing PASS/FAIL/SKIP for each.

    When isolated=True (Level 1), each task gets a fresh explorer so prior
    conversation state does not affect the next task. When isolated=False,
    all tasks share one explorer (useful for Level 2/3 workflow continuity).

    Pass `llm` (e.g. make_llm(model_name)) to benchmark a specific model instead
    of the global default; `model_label` is stamped onto every TaskResult so
    results from different models can be told apart and compared afterward.

    Tasks with requires_network are skipped when LEVELS_SKIP_NETWORK=1. Any task
    that raises is caught and recorded as a failed TaskResult instead of aborting
    the rest of the sweep — this includes TaskTimeoutError when a task exceeds
    its timeout_sec / TASK_TIMEOUT_SEC, and other errors (e.g. a model that
    errors on tool-calling).
    """
    results = []
    explorer = None if isolated else make_explorer(llm=llm)
    for task in tasks:
        if task.requires_network and os.environ.get("LEVELS_SKIP_NETWORK", "0") == "1":
            task = BenchmarkTask(**{**asdict(task), "skip": True})
        task_explorer = make_explorer(llm=llm) if isolated else explorer
        task_timeout = task.timeout_sec if task.timeout_sec is not None else TASK_TIMEOUT_SEC
        try:
            tr = run_task(task_explorer, task)
            tr = score_task(task, tr)
        except TaskTimeoutError as e:
            tr = TaskResult(
                task_id=task.task_id, level=task.level, category=task.category, query=task.query,
                response="", tool_calls=[], elapsed_sec=task_timeout, tokens=0,
                passed=False, score=0.0, notes=f"timeout: {e}",
            )
        except Exception as e:
            tr = TaskResult(
                task_id=task.task_id, level=task.level, category=task.category, query=task.query,
                response="", tool_calls=[], elapsed_sec=0.0, tokens=0,
                passed=False, score=0.0, notes=f"error: {e!r}",
            )
        tr.model = model_label
        results.append(tr)
        prefix = f"{model_label} | " if model_label else ""
        if tr.notes == "skipped":
            status = "SKIP"
        elif tr.notes.startswith("timeout:"):
            status = "TIMEOUT"
        elif tr.passed:
            status = "PASS"
        else:
            status = "FAIL"
        print(f"[{status}] {prefix}{tr.task_id} — score={tr.score}")
        if tr.timing_log:
            print(format_timing_summary(tr.timing_log))
        print()
    return results


def results_to_dataframe(results: List[TaskResult], tasks: Optional[List[BenchmarkTask]] = None) -> pd.DataFrame:
    """Flatten TaskResult objects into a summary DataFrame for display and analysis.

    Pass the corresponding `tasks` list to additionally compute RQ1 productivity columns
    (time_ratio / effective_productivity) for any task with reference_time_sec set.
    """
    task_by_id = {t.task_id: t for t in (tasks or [])}
    rows = []
    for r in results:
        task = task_by_id.get(r.task_id)
        row = {
            "model": r.model,
            "task_id": r.task_id,
            # Prefer the result's own `query` (the actual text sent, e.g. a variant's
            # wording) over the task lookup, which may only reflect the base/standard phrasing.
            "query": r.query or (task.query if task is not None else ""),
            "prompt_style": getattr(task, "prompt_style", "standard") if task is not None else "standard",
            "base_task_id": (getattr(task, "base_task_id", "") or r.task_id) if task is not None else r.task_id,
            "level": r.level,
            "category": r.category,
            "passed": r.passed,
            "score": r.score,
            "elapsed_sec": round(r.elapsed_sec, 1),
            "tokens": r.tokens,
            "tools": ", ".join(t["name"] for t in r.tool_calls),
            "response": r.response,
            "notes": r.notes,
            "expects_abstention": task_by_id[r.task_id].expects_abstention if r.task_id in task_by_id else None,
            "hallucination_detected": r.notes.startswith("HALLUCINATION") if r.notes else False,
            "checkpoint_opportunity_count": len(r.checkpoint_events),
            "high_stakes_action_count": sum(1 for e in r.checkpoint_events if e.get("type") == "high_stakes_action"),
            "hedge_language_count": sum(1 for e in r.checkpoint_events if e.get("type") == "hedge_language"),
            "failure_category": r.failure_category,
        }
        task = task_by_id.get(r.task_id)
        if task is not None and task.reference_time_sec:
            row["reference_time_sec"] = task.reference_time_sec
            row["time_ratio"] = round(r.elapsed_sec / task.reference_time_sec, 2)
            row["productivity_gain"] = round(1 - row["time_ratio"], 2)
        if r.check_results:
            for k, v in r.check_results.items():
                row[f"check_{k}"] = v
        if r.judge_scores:
            for k, v in r.judge_scores.items():
                if isinstance(v, (int, float, str, bool)):
                    row[f"judge_{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


def timing_breakdown_dataframe(results: List[TaskResult]) -> pd.DataFrame:
    """Flatten every TaskResult's timing_log into one long-format DataFrame (one row per
    recorded phase per task), for aggregate "where does time go" analysis across a whole
    benchmark run -- e.g.:

        tdf = timing_breakdown_dataframe(l2_results)
        tdf.groupby("phase")["elapsed_sec"].sum().sort_values(ascending=False)
        tdf.groupby(["model", "phase"])["elapsed_sec"].sum().unstack("phase")

    Columns: model, task_id, level, phase, phase_group (phase split on ':' -- e.g. "tool",
    "score", "step", "llm_call" -- for coarse grouping independent of which tool/step/score
    stage it was), is_nested (True for llm_call/tool:* entries, which are recorded *inside* a
    step:*/setup_query invocation and are already counted in that phase's own elapsed_sec --
    see tool_utils._is_nested_phase), scope (which step:*/setup_query a nested entry happened
    during), elapsed_sec, error (if the phase raised).

    IMPORTANT: summing elapsed_sec across phase_group (or phase) double-counts nested entries
    on top of the step:*/setup_query phase that already contains them. For a real wall-clock
    total, filter to `~is_nested` first (or use tool_utils.wall_clock_total() per-task).
    """
    rows = []
    for r in results:
        for entry in r.timing_log:
            phase = entry.get("phase", "")
            rows.append({
                "model": r.model,
                "task_id": r.task_id,
                "level": r.level,
                "phase": phase,
                "phase_group": phase.split(":", 1)[0],
                "is_nested": _is_nested_phase(phase),
                "scope": entry.get("scope", ""),
                "elapsed_sec": entry.get("elapsed_sec", 0.0),
                "error": entry.get("error"),
            })
    return pd.DataFrame(
        rows,
        columns=["model", "task_id", "level", "phase", "phase_group", "is_nested", "scope", "elapsed_sec", "error"],
    )


def effective_productivity(df: pd.DataFrame) -> pd.DataFrame:
    """RQ1: 'did agentic assistance improve productivity?' A task only counts as a
    productivity win if the agent BOTH passed it AND finished faster than the human
    reference_time_sec baseline -- a fast wrong answer doesn't count. Requires
    reference_time_sec to have been populated on the tasks (see BenchmarkTask docstring);
    returns an empty frame with a warning if no task has that field set yet.
    """
    if "time_ratio" not in df.columns or df["time_ratio"].dropna().empty:
        print(
            "No reference_time_sec set on any task yet -- effective_productivity requires a "
            "human to time execution of each task's reference_path once. See "
            "BENCHMARK_METHODOLOGY.md (RQ1) for the data-collection protocol."
        )
        return pd.DataFrame()
    sub = df.dropna(subset=["time_ratio"]).copy()
    return (
        sub.groupby("model")
        .apply(lambda g: pd.Series({
            "pass_rate": g["passed"].mean(),
            "mean_productivity_gain_when_passed": g.loc[g["passed"] == True, "productivity_gain"].mean(),
            "effective_productivity": g["passed"].mean() * g.loc[g["passed"] == True, "productivity_gain"].mean(),
            "n_tasks": len(g),
        }))
        .round(3)
    )


def _current_git_commit() -> str:
    """Best-effort short git commit hash for reproducibility metadata; '' if unavailable."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"], cwd=str(AI_SEARCH_DIR),
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip() if out.returncode == 0 else ""
    except Exception:
        return ""


def save_results(results: List[TaskResult], label: str) -> Path:
    """Write benchmark results to levels_benchmark_results/{RUN_ID}_{label}.json, plus a
    sibling {RUN_ID}_{label}.meta.json with run-level reproducibility metadata (git commit,
    timeout/MCP config) that isn't repeated per-task -- addresses the "no run metadata"
    reproducibility gap called out in BENCHMARK_METHODOLOGY.md.
    """
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out = RESULTS_DIR / f"{RUN_ID}_{label}.json"
    payload = [asdict(r) for r in results]
    out.write_text(json.dumps(payload, indent=2, default=str))

    meta_out = RESULTS_DIR / f"{RUN_ID}_{label}.meta.json"
    meta = {
        "run_id": RUN_ID,
        "label": label,
        "git_commit": _current_git_commit(),
        "task_timeout_sec": TASK_TIMEOUT_SEC,
        "use_mcp": USE_MCP,
        "master_db": MASTER_DB,
        "n_tasks": len(results),
    }
    meta_out.write_text(json.dumps(meta, indent=2, default=str))
    print(f"Saved: {out}")
    return out

def format_dsi_call_log(dsi_calls: List[Dict[str, Any]]) -> str:
    """Pretty-print recorded DSI Python API calls (method, args, kwargs, timing)."""
    if not dsi_calls:
        return "(no DSI API calls)"
    lines = []
    for i, call in enumerate(dsi_calls, 1):
        lines.append(
            f"[{i}] DSI.{call['method']}("
            f"{', '.join(call.get('args', []))}"
            f"{', ' if call.get('args') and call.get('kwargs') else ''}"
            f"{', '.join(f'{k}={v}' for k, v in call.get('kwargs', {}).items())})"
            f"  [{call.get('elapsed_sec', 0):.3f}s]"
            + (f"  ERROR: {call['error']}" if call.get("error") else "")
        )
    return "\n".join(lines)


## Task Catalog

Tasks are drawn from the example queries in `dsi_data_explorer.ipynb`, `dsi_data_explorer-poisson.ipynb`, and `dsi_code_run.ipynb`.

In [7]:
# Level 1 — Atomic (single tool call, binary pass/fail + quantitative checks)
# Load data, store data, query data, extract data, search data, download data
#
# Each task's `reference_path` is the most-efficient sequence of DSI CLI commands a human
# would run to answer the same query. score_efficiency() LLM-judges the agent's actual
# tool_calls/dsi_calls against this reference to produce a judge_efficiency_score,
# independent of the binary correctness score from score_level1.

LEVEL1_TASKS = [
    BenchmarkTask(
        task_id="L1_load_ocean11",
        level=1, category="Data Access",
        query="Load the ocean_11_datasets.db database.",
        prompt_variants={
            "novice": "Hey, can you open up that ocean_11_datasets.db file for me? I just want to see what's in it.",
            "expert": "Instantiate a DSI load operation against the ocean_11_datasets.db backend so its schema/context is available for subsequent queries.",
            "terse": "load ocean_11_datasets.db",
            "verbose": "I'm working on a project that requires me to explore several scientific datasets, and the first one I need to look at is stored in a file called ocean_11_datasets.db -- could you please go ahead and load that database for me so that I can start examining its contents?",
            "casual": "yo can u load up ocean_11_datasets.db real quick, thx",
        },
        expected_tools=["load_dsi_tool"],
        expected_path= "oceans_11/ocean_11_datasets.db",
        expected_args={"path": "oceans_11/ocean_11_datasets.db"},
        reference_path=["read /Users/hhn/lanl/dsi/tools/ai_search/oceans11.db"],
    ),
    BenchmarkTask(
        task_id="L1_load_nif_direct",
        level=1, category="Data Access",
        query="Load the nif.db database.",
        prompt_variants={
            "novice": "Can you open the nif.db file so I can look at the NIF data?",
            "expert": "Perform a DSI load against the nif.db backend to bring its schema into context.",
            "terse": "load nif.db",
            "verbose": "For the next part of my analysis I specifically need access to the National Ignition Facility simulation data, which I believe is stored in a database file named nif.db -- would you be able to load that one for me now?",
            "casual": "can u pull up nif.db for me",
        },
        expected_tools=["load_dsi_tool"],
        expected_path="oceans_11/nif.db",
        expected_args={"path": "oceans_11/nif.db"},
        reference_path=["read data/oceans_11/nif.db"],
    ),
    BenchmarkTask(
        task_id="L1_load_poisson",
        level=1, category="Data Access",
        query="Load the poisson.db database.",
        prompt_variants={
            "novice": "Please open the poisson.db file.",
            "expert": "Execute a load operation on the poisson.db backend.",
            "terse": "load poisson.db",
            "verbose": "I'd like to begin looking at the electrostatics simulation results, which I understand live in a database called poisson.db -- could you load that database so I can start digging into it?",
            "casual": "load poisson.db pls",
        },
        expected_tools=["load_dsi_tool"],
        expected_path="oceans_11/poisson.db",
        expected_args={"path": "oceans_11/poisson.db"},
        reference_path=["read data/oceans_11/poisson.db"],
    ),
    BenchmarkTask(
        task_id="L1_search_implosion",
        level=1, category="Data Discovery",
        query="Do you have any implosion data in oceans 11 database?",
        prompt_variants={
            "novice": "I'm looking for anything about things imploding, like in a physics experiment. Got anything like that in the oceans 11 database?",
            "expert": "Query the Ocean 11 catalog for any dataset whose metadata indicates implosion-driven physics (e.g. inertial confinement fusion).",
            "terse": "implosion data in oceans11?",
            "verbose": "As part of a broader survey of high energy density physics datasets, I'm trying to determine whether the Ocean 11 catalog contains any resource related to implosion phenomena -- for instance, simulations of capsule or target implosions -- could you check and let me know what you find?",
            "casual": "yo is there any implosion stuff in the oceans 11 db",
        },
        expected_tools=["query_dsi_tool"],
        response_keywords=["National Ignition", "NIF", "FLASH"],
        forbidden_keywords=["Bowtie", "Deep Water Impact"],
        reference_path=[
            "read /Users/hhn/lanl/dsi/tools/ai_search/oceans11.db",
            "search oceans11 implosion",
        ],
    ),
    BenchmarkTask(
        task_id="L1_download_nif_sim",
        level=1, category="Data Retrieval",
        query="Download the simulation data for sim name: nifcylxyz_hdf5_plt_cnt_0000.flash",
        prompt_variants={
            "novice": "Can you grab the data file for the simulation called nifcylxyz_hdf5_plt_cnt_0000.flash?",
            "expert": "Resolve and retrieve the dataset artifact corresponding to sim_name='nifcylxyz_hdf5_plt_cnt_0000.flash' via download_file_tool.",
            "terse": "download nifcylxyz_hdf5_plt_cnt_0000.flash",
            "verbose": "I need to get my hands on the actual simulation output for one specific run -- the one identified by the simulation name nifcylxyz_hdf5_plt_cnt_0000.flash -- so could you please locate and download that particular file for me?",
            "casual": "can u download the file for nifcylxyz_hdf5_plt_cnt_0000.flash",
        },
        expected_tools=["download_file_tool"],
        expected_args={"url": "https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash"},
        requires_network=True,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
        ],
    ),
    BenchmarkTask(
        task_id="L1_store_poisson_hdf5",
        level=1, category="Data Retrieval",
        query="Download this simulation data file: https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
        prompt_variants={
            "novice": "Can you download this file for me? https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
            "expert": "Issue a retrieval of the HDF5 artifact at https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
            "terse": "download https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
            "verbose": "I came across this link to what I believe is an HDF5 simulation data file and I'd like to have a local copy of it to work with -- the URL is https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5 -- could you download it for me?",
            "casual": "can u dl this real quick https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
        },
        expected_tools=["download_file_tool"],
        expected_args={"url": "https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5"},
        requires_network=True,
        reference_path=[
            "pull_data url poisson_hdf5 https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
        ],
    ),
    BenchmarkTask(
        task_id="L1_owner_field",
        level=1, category="Metadata Query",
        query=f'Who is the owner of the NIF dataset"',
        prompt_variants={
            "novice": "Who owns the NIF dataset? Like whose data is it?",
            "expert": "Identify the designated point-of-contact/owner of record for the NIF dataset per its catalog metadata.",
            "terse": "NIF dataset owner?",
            "verbose": "I need to know, for attribution and follow-up purposes, exactly who the responsible point of contact or owner is for the NIF dataset that's cataloged in this system -- could you tell me who that is?",
            "casual": "who owns the nif dataset lol",
        },
        expected_tools=["query_dsi_tool"],
        response_keywords=["Joshua", "Paul", "Sauppe"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, Email FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_dataset_count",
        level=1, category="Data Exploration",
        query="How many datasets are in the Ocean 11 catalog?",
        prompt_variants={
            "novice": "How many datasets does the Ocean 11 catalog have?",
            "expert": "Return the cardinality of the dataset collection in the Ocean 11 catalog (i.e. row count of the genesis_datacard table).",
            "terse": "# datasets in oceans11?",
            "verbose": "For a summary report I'm putting together, I need to state precisely how many individual datasets are currently cataloged within the Ocean 11 system -- could you tell me that total count?",
            "casual": "how many datasets we got in oceans 11",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=9,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT COUNT(*) FROM genesis_datacard"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_originating_org",
        level=1, category="Metadata Query",
        query="What organization is listed for the Ocean 11 datasets?",
        prompt_variants={
            "novice": "Which organization made the Ocean 11 datasets?",
            "expert": "Report the Originating_Organization field value(s) associated with the Ocean 11 catalog records.",
            "terse": "oceans11 org?",
            "verbose": "I'm trying to confirm, for a compliance note, which organization is officially listed as the originating body responsible for the datasets held in the Ocean 11 catalog -- can you tell me what that organization is?",
            "casual": "who's org is behind oceans 11 datasets",
        },
        expected_tools=["query_dsi_tool"],
        response_keywords=["DOE", "LANL"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT DISTINCT Originating_Organization FROM genesis_datacard"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_physics_theme_count",
        level=1, category="Metadata Query",
        query="How many datasets are categorized as physics?",
        prompt_variants={
            "novice": "How many of the datasets are about physics?",
            "expert": "Count the number of catalog records whose Theme/Category field equals 'physics'.",
            "terse": "# physics datasets?",
            "verbose": "I'm trying to narrow my search to just the physics-related holdings in this catalog, so as a first step could you tell me exactly how many datasets are categorized under the physics theme?",
            "casual": "how many of these are physics ones",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=5,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT COUNT(*) FROM genesis_datacard WHERE LOWER(\\\"Theme/Category_:_Domain\\\") = 'physics'\"",
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_domain_extent",
        level=1, category="Data Exploration",
        query="What are the spatial bounds of the NIF simulation's computational domain?",
        prompt_variants={
            "novice": "How big is the area that the NIF simulation covers, like the physical size of the domain?",
            "expert": "Report the spatial extent (bounding dimensions) of the computational domain used in the NIF simulation.",
            "terse": "NIF domain size?",
            "verbose": "I need to understand the physical scale of the simulated region for the NIF run -- specifically, what are the spatial bounds or dimensions of its computational domain -- so I can compare it against another simulation's setup?",
            "casual": "how big is the nif sim domain",
        },
        expected_tools=["query_dsi_tool"],
        response_keywords=["3.072"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "search computational domain",
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_timestep_count",
        level=1, category="Data Exploration",
        query="How many discrete timesteps of the NIF simulation are recorded in NIF.",
        prompt_variants={
            "novice": "How many time steps does the NIF simulation have saved?",
            "expert": "Report the count of distinct timestep values recorded in the NIF simulation metadata.",
            "terse": "# NIF timesteps?",
            "verbose": "For my timeline analysis, I need to know exactly how many distinct timesteps were recorded over the course of the NIF simulation -- could you look that up for me?",
            "casual": "how many timesteps we got for nif",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=20,
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_array_field_count",
        level=1, category="Data Exploration",
        query="How many distinct physical quantity arrays are recorded per simulation timestep in NIF.",
        prompt_variants={
            "novice": "How many different measurements does NIF record at each timestep?",
            "expert": "Report the number of distinct physical-quantity arrays captured per timestep in the NIF simulation (per array_and_types).",
            "terse": "# arrays per NIF timestep?",
            "verbose": "I'm trying to understand the breadth of the NIF dataset's physical measurements -- specifically, at each individual timestep, how many distinct physical quantity arrays (e.g. density, pressure, etc.) are being recorded?",
            "casual": "how many kinds of arrays does nif track per step",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=9,
        response_keywords=["dens", "pres"],
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT * FROM array_and_types"',
        ],
    ),
    # cd tools/ai_search
    # dsi
    # dsi> search dens
    # dsi> search dens_max *returns no results*
    # Too complicated for level 1
    # BenchmarkTask(
    #     task_id="L1_dens_max_units",
    #     level=1, category="Metadata Generation",
    #     query="What unit is dens_max recorded in?",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["g/cm"],
    # ),

]


### Level 2 — Workflow Tasks

Multi-step pipelines combining ≥2 data-management categories (e.g. search → download →
summarize, filter → report, ingest → validate). Run **non-isolated** (see `run_level`), so
tasks sharing a list may see each other's conversational context; tasks that need guaranteed
clean state use `setup_query` or an explicit `steps` list of turns run against one shared
explorer (`task.steps`, see `run_task`). Scored by `score_level2()`: every tool in
`expected_tools` must be used at least once (not any-of, like Level 1), plus an LLM-judged
process-quality score against `success_criteria`. Ground-truth counts/fields below were
hand-verified against `data/oceans_11/*.db` directly with sqlite3, the same way Level 1's
were verified via the DSI CLI.

#### How `score_level2()` scores a task

| Component | What it checks | Pass condition | Contributes to |
|---|---|---|---|
| **Tool coverage** (`tools_ok`) | Did the agent call *every* tool listed in `expected_tools`? | ALL required tools must appear in `tool_calls` (unlike Level 1, where only *one* correct tool is needed) | Correctness |
| **Quantitative check** (`quant_ok`) | Does the expected number (`expected_count`) show up in the tool results or final response? | Regex match for the number as a `count:`/`missing_count:` field or a standalone number in the combined text blob | Correctness |
| **Keyword check** (`keyword_ok`) | Do all required facts (`response_keywords`) appear in the response/tool output? | ALL keywords must be present (case-insensitive) — every one, not just any one | Correctness |
| **Forbidden-keyword check** (`forbidden_ok`) | Did the agent avoid stating any wrong/distractor facts (`forbidden_keywords`)? | None of the forbidden terms may appear | Correctness |
| **Non-empty check** (`nonempty`) | Did the agent actually produce a final answer? | `response` must be non-blank | Correctness |
| **Correctness** (`result.passed`) | All five checks above | `tools_ok AND quant_ok AND keyword_ok AND forbidden_ok AND nonempty` | Binary 0.0 / 1.0 — **50% of final score** |
| **Process quality** (`judge_scores["process_quality_score"]`) | Was the *method* good — efficient, well-ordered, matching the `success_criteria` rubric — regardless of whether the final answer was right? | LLM judge (both models, scores averaged) rates 0.0 (ignored rubric / haphazard) to 1.0 (clean, sound multi-step approach) | Continuous 0.0–1.0 — **50% of final score** |

**Final score formula:**
- If the task has `success_criteria` set: `score = 0.5 × correctness (0 or 1) + 0.5 × process_quality_score`
- If no `success_criteria`: `score = correctness` only (0.0 or 1.0)

**Important nuance:** `result.passed` reflects **correctness only** — a task can "pass" (did all the right steps, got the right facts) while still scoring well below 1.0 overall if the LLM judge dings it for a messy/inefficient process. That's deliberate: it lets the benchmark distinguish "got it right, cleanly" from "got it right, but wastefully."

In [8]:
LEVEL2_TASKS = [
    # query catalog -> download file -> query a second DB (2 tools, cross-file)
    BenchmarkTask(
        task_id="L2_search_download_summarize",
        level=2, category="Analysis & Summarization",
        query=(
            "Find the NIF simulation dataset in the Ocean 11 catalog, download the file for "
            "sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset "
            "and how many discrete timesteps the simulation has."
        ),
        prompt_variants={
            "novice": "Can you find the NIF dataset in the Ocean 11 catalog, grab the file for the simulation named nifcylxyz_hdf5_plt_cnt_0000.flash, and then tell me who owns it and how many timesteps it has?",
            "expert": "Resolve the NIF dataset record in the Ocean 11 catalog, retrieve the artifact for sim_name=nifcylxyz_hdf5_plt_cnt_0000.flash via download_file_tool, then report the dataset's point-of-contact and the distinct timestep count.",
            "terse": "find NIF dataset, download nifcylxyz_hdf5_plt_cnt_0000.flash, report owner + timestep count",
            "verbose": "As part of a multi-step data provenance exercise, I first need you to locate the NIF simulation dataset within the Ocean 11 catalog; once you've found it, please download the specific simulation file associated with the sim name nifcylxyz_hdf5_plt_cnt_0000.flash; and finally, report back to me both who is listed as the owner of that dataset and how many discrete timesteps the overall simulation contains.",
            "casual": "yo find the nif dataset in oceans 11, dl the file for nifcylxyz_hdf5_plt_cnt_0000.flash, then lmk who owns it and how many timesteps it's got",
        },
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["Sauppe", "20"],
        requires_network=True,
        success_criteria=(
            "Should query the Ocean 11 catalog to find the NIF dataset's DB link, download the "
            "specified simulation file, and query nif.db to report both the owner's name "
            "(Joshua Paul Sauppe) and the timestep count (20) without redundant re-exploration."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),
    # single-table filter + frequency aggregation (1 tool)
    BenchmarkTask(
        task_id="L2_physics_contact_report",
        level=2, category="Metadata Generation",
        query=(
            "List the titles of every physics-themed dataset in the Ocean 11 catalog "
            "(case-insensitive) along with each one's point of contact, and tell me which "
            "contact person appears most often among them."
        ),
        prompt_variants={
            "novice": "Can you list all the physics datasets in Ocean 11 along with who's in charge of each one, and tell me which person shows up the most?",
            "expert": "Enumerate all catalog records where Theme/Category_:_Domain matches 'physics' (case-insensitive), list each record's Contact_Point_, and identify the most frequently occurring contact.",
            "terse": "list physics datasets + contacts, who's most frequent?",
            "verbose": "I'm compiling a stakeholder report and need the complete list of titles for every dataset in the Ocean 11 catalog that falls under the physics theme or category, regardless of capitalization, along with the point of contact listed for each one; additionally, once you have that list, please identify which single contact person is associated with the largest number of those physics datasets.",
            "casual": "can u list out all the physics datasets in oceans11 plus their contacts and tell me who pops up most",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=5,
        response_keywords=["Nathan"],
        success_criteria=(
            "Should identify all 5 physics-themed datasets (case-insensitive match on the theme "
            "field: Deep Water Impact, Higrad Firetex Wildfire, Gray-Scott reaction-diffusion, "
            "Heat Equations, and Poisson's Equations for Electrostatics), list their contacts, "
            "and correctly identify that Nathan DeBardeleben (or a near-identical spelling "
            "variant on the Poisson record) is associated with the most of them (3 of 5: "
            "Gray-Scott, Heat Equations, and Poisson)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_ FROM genesis_datacard WHERE LOWER(\\\"Theme/Category_:_Domain\\\") = 'physics'\"",
        ],
    ),
    # load + inspect every field of one row for completeness (2 tools)
    BenchmarkTask(
        task_id="L2_poisson_field_completeness",
        level=2, category="Data Preparation",
        query=(
            "Load the Poisson database and check whether its single record has a value in "
            "every field. Report the number of empty, null, or missing fields you find."
        ),
        prompt_variants={
            "novice": "Can you open the Poisson database and check if its one record has everything filled in, or if anything's missing?",
            "expert": "Load the poisson.db backend and perform a completeness audit on its single record across all fields, reporting the count of null/empty/missing values.",
            "terse": "load poisson.db, check record for missing fields, report count",
            "verbose": "I'd like you to load the Poisson database and then carefully go through its single data record field by field to determine whether every field actually contains a value -- once you've done that, please report back the total number of fields you found to be empty, null, or otherwise missing data.",
            "casual": "can u load poisson.db and check if any fields on that one record are empty or missing, just gimme the count",
        },
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        expected_count=0,
        success_criteria=(
            "Should load poisson.db, inspect the single record across all 7 fields (name, LAUR, "
            "example, environment, argements, data, statistics), and correctly report that none "
            "are missing/empty/null -- all seven contain populated URLs."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            'query "SELECT * FROM poisson"',
        ],
    ),
    #3-turn conversation testing memory across turns (load, load, compare)
    BenchmarkTask(
        task_id="L2_cross_db_record_compare",
        level=2, category="Analysis & Summarization",
        query="Compare the number of discrete records in the NIF dataset vs. the Poisson dataset.",
        steps=[
            "Load nif.db and tell me how many discrete timesteps are recorded.",
            "Now load poisson.db and tell me how many records it contains.",
            "Of the two datasets you just examined, which has more discrete records, and by how many?",
        ],
        prompt_variants={
            "novice": [
                "Can you open up nif.db and tell me how many different timesteps it has?",
                "Okay now can you open poisson.db and tell me how many records are in it?",
                "Between those two you just looked at, which one has more records, and how many more?",
            ],
            "expert": [
                "Load the nif.db backend and report the cardinality of distinct timestep values.",
                "Now load the poisson.db backend and report its total record count.",
                "Compare the two record counts just obtained and report which dataset has the greater cardinality, and the magnitude of the difference.",
            ],
            "terse": [
                "load nif.db, # timesteps?",
                "load poisson.db, # records?",
                "which has more, by how much?",
            ],
            "verbose": [
                "To begin this comparison exercise, could you please load the nif.db database and tell me precisely how many discrete timesteps are recorded within it?",
                "Great, now building on that, could you also load the separate poisson.db database and tell me exactly how many records it contains in total?",
                "Now that you've examined both of those databases, could you tell me which of the two contains a greater number of discrete records, and specifically by what margin?",
            ],
            "casual": [
                "yo load up nif.db real quick and tell me how many timesteps it's got",
                "now do the same for poisson.db, how many records in that one",
                "so which one's got more records, and by how many",
            ],
        },
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        expected_count=19,
        response_keywords=["nif", "poisson"],
        success_criteria=(
            "Should recall (or re-verify) NIF's 20 timesteps and Poisson's 1 record from earlier "
            "in this same conversation and correctly conclude NIF has 19 more discrete records "
            "than Poisson, ideally without wastefully re-loading/re-querying data it already "
            "has the answer for."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
            "read data/oceans_11/poisson.db",
            'query "SELECT COUNT(*) FROM poisson"',
        ],
    ),
    # full-table scan + counting a missing field (1 tool)
    BenchmarkTask(
        task_id="L2_dsi_database_link_audit",
        level=2, category="Metadata Generation",
        query=(
            "For every dataset in the Ocean 11 catalog, check whether it has a linked DSI "
            "sub-database (the dsi_database field) and report how many of the 9 datasets are "
            "missing one."
        ),
        prompt_variants={
            "novice": "Can you check all the datasets in Ocean 11 and tell me how many of the 9 don't have a linked DSI sub-database?",
            "expert": "Audit the dsi_database field across all 9 genesis_datacard records and report the count of records with a null/missing linked sub-database.",
            "terse": "# of 9 oceans11 datasets missing dsi_database link?",
            "verbose": "I'm conducting a data-completeness audit and need to go through every single dataset in the Ocean 11 catalog -- all 9 of them -- checking whether each one has an associated/linked DSI sub-database recorded in its dsi_database field, and then report back the total number of datasets that are missing that link.",
            "casual": "can u check all 9 oceans11 datasets and tell me how many don't have a dsi_database link",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=7,
        success_criteria=(
            "Should query the dsi_database field across all 9 genesis_datacard records and "
            "correctly report that 7 of the 9 datasets have no linked DSI sub-database (only "
            "the NIF and Poisson entries do)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT Title, dsi_database FROM genesis_datacard"',
        ],
    ),
    # query -> download -> extract facts from a filename (2 tools)
    BenchmarkTask(
        task_id="L2_poisson_args_filename_facts",
        level=2, category="Data Preparation",
        query=(
            "Download the Poisson dataset's simulation-arguments file (the 'argements' field "
            "in the Poisson catalog record) and, from its filename or contents, report the "
            "grid size and number of samples in this dataset."
        ),
        prompt_variants={
            "novice": "Can you download the arguments file for the Poisson dataset and tell me the grid size and how many samples there are, based on the file?",
            "expert": "Retrieve the simulation-arguments artifact referenced by the Poisson record's 'argements' field, and extract the grid resolution and sample count from its filename/contents.",
            "terse": "download poisson 'argements' file, report grid size + sample count",
            "verbose": "I need you to download the file referenced by the 'argements' field on the Poisson catalog record -- this should be the simulation-arguments file -- and then, based on either its filename or its actual contents, report back to me both the grid size used for the simulation and the total number of samples included in the dataset.",
            "casual": "can u dl the poisson args file and tell me the grid size + sample count from it",
        },
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["32x32", "1000"],
        requires_network=True,
        success_criteria=(
            "Should query poisson.db for the 'argements' URL, download it, and correctly "
            "extract '32x32' as the grid size and '1000' (from the 1-1000 sample range in the "
            "filename/contents) as the sample count, without fabricating different numbers."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            'query "SELECT argements FROM poisson"',
            "pull_data url poisson_args https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/arguments_electrostatic_poisson_32x32_1-1000.json",
        ],
    ),
    #cross-referencing a static doc file against live schema, not just querying one source
    BenchmarkTask(
        task_id="L2_poisson_glossary_audit_1",
        level=2, category="Data Preparation",
        query=(
            "The Poisson dataset has a field glossary file at "
            "data/oceans_11/poisson_description.yaml describing its documented columns. "
            "Load poisson.db, then read the glossary file and compare the ACTUAL table "
            "schema against the fields documented in the glossary. Report any column that "
            "exists in the real data but is NOT documented in the glossary."
        ),
        prompt_variants={
            "novice": "There's a file at data/oceans_11/poisson_description.yaml that's supposed to describe all the columns in poisson.db. Can you load poisson.db, check that glossary file, and tell me if there's any column in the real data that's missing from the glossary?",
            "expert": "Load the poisson.db schema and cross-reference it against the field definitions documented in data/oceans_11/poisson_description.yaml; report any schema column absent from that glossary.",
            "terse": "load poisson.db, read data/oceans_11/poisson_description.yaml, diff schema vs glossary fields",
            "verbose": "I'd like you to perform a documentation-completeness check: first load the poisson.db database so you can see its actual table schema, then separately open and read the field glossary file located at data/oceans_11/poisson_description.yaml, which is supposed to document every column in that table -- once you've done both, compare the two lists and report any column that genuinely exists in the real database but isn't documented anywhere in that glossary file.",
            "casual": "can u load poisson.db, check the glossary at data/oceans_11/poisson_description.yaml, and tell me if any real column isn't in it",
        },
        expected_tools=["load_dsi_tool", "python_repl_tool"],
        response_keywords=["argements"],
        success_criteria=(
            "Should inspect poisson.db's real schema (name, LAUR, example, environment, "
            "argements, data, statistics), compare it field-by-field against the glossary's "
            "documented fields (name, LAUR, example, environment, data, statistics), and "
            "correctly identify 'argements' as the one column present in the data but "
            "undocumented in the glossary -- without inventing a different missing field or "
            "claiming the schema and glossary fully match."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            "schema",
            "read data/oceans_11/poisson_description.yaml",
            "diff schema fields vs glossary fields",
        ],
    ),

        #cross-referencing a static doc file against live schema, not just querying one source
    BenchmarkTask(
        task_id="L2_poisson_glossary_audit_2",
        level=2, category="Data Preparation",
        query=(
            "The Poisson dataset has a field glossary (poisson_description.yaml) describing its columns as: name, LAUR, example, environment, data, statistics. Load poisson.db, compare its ACTUAL table schema against that documented list of fields, and report any column that exists in the real data but is NOT documented in the glossary."
        ),
        prompt_variants={
            "novice": "There's a file at data/oceans_11/poisson_description.yaml that's supposed to describe all the columns in poisson.db. Can you load poisson.db, check that glossary file, and tell me if there's any column in the real data that's missing from the glossary?",
            "expert": "Load the poisson.db schema and cross-reference it against the field definitions documented in data/oceans_11/poisson_description.yaml; report any schema column absent from that glossary.",
            "terse": "load poisson.db, read data/oceans_11/poisson_description.yaml, diff schema vs glossary fields",
            "verbose": "I'd like you to perform a documentation-completeness check: first load the poisson.db database so you can see its actual table schema, then separately open and read the field glossary file located at data/oceans_11/poisson_description.yaml, which is supposed to document every column in that table -- once you've done both, compare the two lists and report any column that genuinely exists in the real database but isn't documented anywhere in that glossary file.",
            "casual": "can u load poisson.db, check the glossary at data/oceans_11/poisson_description.yaml, and tell me if any real column isn't in it",
        },
        expected_tools=["load_dsi_tool", "python_repl_tool"],
        response_keywords=["argements"],
        success_criteria=(
            "Should inspect poisson.db's real schema (name, LAUR, example, environment, "
            "argements, data, statistics), compare it field-by-field against the glossary's "
            "documented fields (name, LAUR, example, environment, data, statistics), and "
            "correctly identify 'argements' as the one column present in the data but "
            "undocumented in the glossary -- without inventing a different missing field or "
            "claiming the schema and glossary fully match."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            "schema",
            "read data/oceans_11/poisson_description.yaml",
            "diff schema fields vs glossary fields",
        ],
    ),

    
    # tests free-text/substring filtering on an unstructured tag field,
    BenchmarkTask(
        task_id="L2_simulation_tag_contacts",
        level=2, category="Metadata Generation",
        query=(
            "Find every dataset in the Ocean 11 catalog whose keywords/tags mention "
            "'simulation', list each one's point-of-contact owner, and tell me if any "
            "contact owns more than one of them."
        ),
        prompt_variants={
            "novice": "Can you find all the datasets in Ocean 11 that have 'simulation' somewhere in their tags, list who owns each one, and tell me if anyone owns more than one?",
            "expert": "Filter the Ocean 11 catalog for records whose Keywords/Tag field contains the substring 'simulation', list each record's Contact_Point_, and identify any contact appearing on more than one matching record.",
            "terse": "find datasets tagged 'simulation', list contacts, any contact own >1?",
            "verbose": "I'm trying to understand ownership concentration among simulation-related holdings, so I need you to search through the Ocean 11 catalog for every dataset whose keywords or tags mention the word 'simulation' in any form, list the point-of-contact owner for each one you find, and then let me know whether any single contact person is responsible for more than one of those datasets.",
            "casual": "can u find all the 'simulation' tagged datasets in oceans11, list their owners, and tell me if anyone's got more than one",
        },
        expected_tools=["query_dsi_tool"],
        expected_count=5,
        response_keywords=["Nathan"],
        success_criteria=(
            "Should find all 5 datasets with 'simulation' in their Keywords/Tag field via "
            "substring/fuzzy matching (Deep Water Impact Ensemble, Higrad Firetex Wildfire, "
            "Gray-Scott reaction-diffusion, Heat Equations, and the 3D FLASH/NIF dataset) -- "
            "not an exact match on the Theme/Category field -- list their contacts, and "
            "correctly report that Nathan DeBardeleben is the only contact appearing more "
            "than once (owns both Gray-Scott and Heat Equations)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_ FROM genesis_datacard WHERE \\\"Keywords/Tag\\\" LIKE '%simulation%'\"",
        ],
    ),
    # requires genuine aggregation/extremum-finding across many rows
    BenchmarkTask(
        task_id="L2_nif_dens_max_extremum",
        level=2, category="Analysis & Summarization",
        query=(
            "Across all 20 timesteps of the NIF simulation, which timestep has the highest "
            "maximum density (dens_max), and what is that value?"
        ),
        prompt_variants={
            "novice": "Looking at all the NIF timesteps, which one has the highest density value, and what's that number?",
            "expert": "Across all nif_metadata rows, identify the timestep with the maximum value of dens_max and report that value.",
            "terse": "highest dens_max across NIF timesteps, which + value?",
            "verbose": "I'm trying to identify the peak density condition reached anywhere during the NIF simulation, so across all 20 recorded timesteps, could you determine which single timestep has the highest value of dens_max, and also tell me what that maximum density value actually is?",
            "casual": "out of all the nif timesteps which one's got the highest dens_max, and what's the number",
        },
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["6", "7.9"],
        success_criteria=(
            "Should aggregate/compare dens_max across ALL 20 nif_metadata rows (e.g. via "
            "MAX()/ORDER BY, or by scanning and comparing every row) rather than reporting a "
            "single arbitrary row, and correctly identify timestep 6 with a dens_max of "
            "approximately 7.95 as the highest."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, dens_max FROM nif_metadata ORDER BY dens_max DESC LIMIT 1"',
        ],
    ),
    #tests whether the agent fabricates a confident-but-wrong distinction when the data actually shows a tie
    BenchmarkTask(
        task_id="L2_nif_num_arrays_tie",
        level=2, category="Analysis & Summarization",
        query=(
            "Which timestep in the NIF simulation has the most data arrays recorded (num_arrays), "
            "and by how many more than the next-highest timestep?"
        ),
        prompt_variants={
            "novice": "Which NIF timestep has the most arrays recorded, and how many more than the next one?",
            "expert": "Identify the timestep with the maximum num_arrays value in nif_metadata, and report the margin over the second-highest value.",
            "terse": "max num_arrays timestep + margin over 2nd?",
            "verbose": "I'm trying to understand whether the amount of data recorded varies across timesteps in the NIF simulation, so could you tell me which single timestep has the greatest number of data arrays recorded (num_arrays), and specifically by how many more arrays than whichever timestep has the next-highest count?",
            "casual": "which nif timestep has the most arrays recorded, and by how much more than the runner up",
        },
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["9"],
        success_criteria=(
            "Should check num_arrays across ALL 20 nif_metadata rows and recognize they are "
            "ALL tied at exactly 9 -- there is no single highest timestep, so the correct "
            "answer explicitly states they're tied (difference of 0), rather than naming an "
            "arbitrary single timestep as 'the winner' or fabricating a nonzero difference."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT DISTINCT num_arrays FROM nif_metadata"',
        ],
    ),
    # a 2-step "find X, then branch on a property of X" workflow where the correct final action (stop, don't drill down) depends on data the agent only discovers mid-task.
    BenchmarkTask(
        task_id="L2_most_recent_drilldown",
        level=2, category="Data Discovery",
        query=(
            "Find the most recently updated dataset in the Ocean 11 catalog, then check "
            "whether it has a linked DSI sub-database (the dsi_database field) that you "
            "could load for more detail."
        ),
        prompt_variants={
            "novice": "Which dataset in Ocean 11 was updated most recently? And does it have a linked database you could open up for more details?",
            "expert": "Identify the catalog record with the most recent Update/Modification_Date, then determine whether its dsi_database field references a loadable sub-database.",
            "terse": "most recently updated oceans11 dataset? has linked dsi_database?",
            "verbose": "I'm trying to figure out which dataset in the Ocean 11 catalog has been updated most recently out of all of them, and once you've identified that dataset, I'd also like you to check whether it has a linked DSI sub-database listed in its dsi_database field that could actually be loaded for additional detail.",
            "casual": "which oceans11 dataset got updated most recently, and does it have a linked db you could dig into",
        },
        expected_tools=["query_dsi_tool"],
        response_keywords=["High Explosives"],
        success_criteria=(
            "Should correctly identify 'The High Explosives & Affected Targets (HEAT) "
            "Dataset' (updated 2025-10-14) as the most recently updated catalog entry, then "
            "correctly report that its dsi_database field is null/empty -- i.e. there is no "
            "linked sub-database to drill into -- rather than fabricating one or incorrectly "
            "naming NIF/Poisson (which DO have linked sub-databases, but are not the most "
            "recently updated)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT Title, dsi_database FROM genesis_datacard ORDER BY \\"Update/Modification_Date\\" DESC LIMIT 1"',
        ],
    ),
]

### Level 3 — Discovery Tasks

Hypothesis-level, DiscoveryBench-style tasks: the agent must analyze real data and state a
finding as a structured hypothesis, scored against `ground_truth_hypothesis` facets
(`context`/`variables`/`relationship`) by a faceted LLM judge (`score_level3`). Every ground
truth below was verified directly against `data/oceans_11/*.db` with sqlite3 (see chat
history / `BENCHMARK_METHODOLOGY.md` for the verification queries) rather than assumed, since
a wrong ground truth would make this an unfalsifiable benchmark.

In [9]:
LEVEL3_TASKS = [
    BenchmarkTask(
        task_id="L3_contact_owner_theme_pattern",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze the Ocean 11 catalog's contact/ownership metadata. Is dataset ownership "
            "concentrated among a small number of contact people, and if so, do those people's "
            "datasets share a common theme? State your finding as a hypothesis with supporting "
            "evidence."
        ),
        ground_truth_hypothesis={
            "context": "Ocean 11 genesis_datacard catalog (9 datasets), the Contact_Point_ field",
            "variables": "Contact_Point_ (owner/point of contact), Theme/Category_:_Domain",
            "relationship": (
                "Ownership is concentrated: 'Nathan DeBardeleben' (with a likely misspelled "
                "variant 'Nathan Debardebelen' on the Poisson record) is the contact for "
                "roughly half the catalog (4-5 of 9 datasets: Bowtie, Gray-Scott, Heat "
                "Equations, Monopoly, and Poisson), but these datasets do NOT share one theme "
                "-- they span both 'manufacturing' (Bowtie, Monopoly) and 'physics' (Gray-Scott, "
                "Heat Equations, Poisson), so ownership concentration does not imply thematic "
                "concentration."
            ),
        },
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Contact_Point_, \\\"Theme/Category_:_Domain\\\" FROM genesis_datacard\"",
        ],
    ),
    BenchmarkTask(
        task_id="L3_nif_density_trend",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze how the NIF simulation's peak density (dens_max) changes across its 20 "
            "recorded timesteps. Characterize the trend and relate it to the physical process "
            "being simulated."
        ),
        ground_truth_hypothesis={
            "context": "nif_metadata table, 20 timesteps (0-19), dens_max column",
            "variables": "timestep, dens_max",
            "relationship": (
                "dens_max rises sharply from about 2.7 at timestep 0 to a peak of about 7.95 "
                "around timestep 6, then declines fairly steadily back down to about 2.6 by "
                "timestep 19 -- a rise-then-fall pattern consistent with compression followed "
                "by rebound/expansion in an implosion simulation, rather than a monotonic "
                "increase or a constant value."
            ),
        },
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, dens_max FROM nif_metadata ORDER BY timestep"',
        ],
    ),
    BenchmarkTask(
        task_id="L3_nif_grid_refinement_trend",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze how the NIF simulation's computational grid (the first number in the "
            "'shape' field) changes across its 20 timesteps. Does the grid resolution stay "
            "fixed, grow, shrink, or fluctuate over the simulation?"
        ),
        ground_truth_hypothesis={
            "context": "nif_metadata table, 20 timesteps (0-19), shape column (first dimension = active zone/cell count)",
            "variables": "timestep, shape (zone count)",
            "relationship": (
                "The zone count is not fixed: it grows overall from 6360 at timestep 0 to 9848 "
                "at timestep 19 (roughly a 55% increase), with some local dips along the way "
                "(e.g. around timesteps 4 and 6-7) rather than perfectly monotonic growth -- "
                "consistent with adaptive mesh refinement tracking the evolving simulation."
            ),
        },
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, shape FROM nif_metadata ORDER BY timestep"',
        ],
    ),
]

### Calibration / Trap Tasks (RQ4 — does the agent know what it doesn't know?)

Each task below references missing data, a nonexistent resource, or a false/ambiguous
premise. The CORRECT behavior is to abstain or ask a clarifying question rather than
confidently fabricate an answer; scored by `score_calibration()` regardless of level via
`expects_abstention=True`. `L1_dens_max_units` reinstates a task that was previously
commented out of `LEVEL1_TASKS` as "too complex" -- `dens_max` genuinely exists as a raw
data column in `nif_metadata` (verified with sqlite3), but the DSI search/description
metadata never documents a *unit* for it, so a well-calibrated agent should say it cannot
determine the unit rather than guessing one (e.g. "g/cm^3").

In [10]:
CALIBRATION_TASKS = [
    BenchmarkTask(
        task_id="L1_dens_max_units",
        level=1, category="Metadata Generation",
        query="What unit is dens_max recorded in?",
        expects_abstention=True,
        # dens_max IS a real column in nif_metadata (verified with sqlite3) but no unit is
        # documented anywhere in the DSI metadata/search index -- correct behavior is to say
        # so, not guess a plausible-sounding unit like "g/cm^3".
    ),
    BenchmarkTask(
        task_id="L1_load_nonexistent_db",
        level=1, category="Data Discovery",
        query="Load the mars_rover.db database.",
        expects_abstention=True,
        # No such file exists anywhere in this workspace -- correct behavior is a clear "file
        # not found" style error, not fabricated schema/content for a Mars rover dataset.
    ),
    BenchmarkTask(
        task_id="L1_nasa_owner_count",
        level=1, category="Metadata Query",
        query="How many Ocean 11 datasets are owned by NASA?",
        expects_abstention=True,
        abstention_keywords=[
            "0", "zero", "none", "no nasa", "not nasa", "no datasets are owned",
            "doe:lanl", "no matching", "not found", "no such",
        ],
        # Verified: Originating_Organization is 'DOE:LANL' for all 9 datasets -- no NASA
        # affiliation exists. Correct behavior is to report zero / that NASA doesn't appear,
        # not fabricate a plausible-sounding positive count.
    ),
    BenchmarkTask(
        task_id="L1_ambiguous_analyze_request",
        level=1, category="Analysis & Summarization",
        query="Analyze the data and tell me what's interesting.",
        expects_abstention=True,
        # No dataset is specified and none has been loaded yet in this fresh (isolated)
        # session -- correct behavior is to ask which dataset to analyze, not guess one.
    ),
    BenchmarkTask(
        task_id="L2_medical_theme_comparison",
        level=2, category="Analysis & Summarization",
        query="Compare the manufacturing-themed datasets in Ocean 11 to the medical-themed datasets.",
        expects_abstention=True,
        abstention_keywords=[
            "no medical", "not medical", "no such theme", "none found", "no datasets",
            "does not include", "there are no medical", "no such", "does not exist",
        ],
        # Verified distinct themes in genesis_datacard: physics, manufacturing, eulerian,
        # fusion -- there is no "medical" theme. This is a false-premise question; correct
        # behavior is to point out no medical-themed datasets exist rather than inventing a
        # comparison against a nonexistent category.
    ),
]

In [11]:
ALL_TASKS = list(LEVEL1_TASKS) + list(LEVEL2_TASKS) + list(LEVEL3_TASKS) + list(CALIBRATION_TASKS)
print(
    f"{len(LEVEL1_TASKS)} Level 1 + {len(LEVEL2_TASKS)} Level 2 + {len(LEVEL3_TASKS)} Level 3 "
    f"+ {len(CALIBRATION_TASKS)} calibration = {len(ALL_TASKS)} total tasks"
)

13 Level 1 + 12 Level 2 + 3 Level 3 + 5 calibration = 33 total tasks


## Run Benchmarks

Set environment variable `LEVELS_SKIP_NETWORK=1` to skip tasks that require external network access.

In [12]:
def run_level(
    label: str,
    tasks: List[BenchmarkTask],
    *,
    llm: Optional[ChatOpenAI] = None,
    model_label: str = "",
) -> List[TaskResult]:
    """Run one complexity level end-to-end: benchmark all tasks and save JSON.

    Pass `llm`/`model_label` to run this level against a specific model (see
    run_benchmark_all_models below to sweep across every model in POSSIBLE_MODELS).
    """
    heading = f"## Running {label}"
    if model_label:
        heading += f" — model: `{model_label}`"
    heading += f" ({len(tasks)} tasks)"
    display(Markdown(heading))
    isolated = all(t.level == 1 for t in tasks)
    results = run_benchmark(tasks, isolated=isolated, llm=llm, model_label=model_label)
    save_label = f"{label}__{_safe_model_label(model_label)}" if model_label else label
    save_results(results, save_label)
    return results


def run_benchmark_all_models(
    tasks: List[BenchmarkTask],
    models: Optional[List[str]] = None,
    level_label: str = "level1_atomic",
) -> Dict[str, List[TaskResult]]:
    """Run the given tasks once per model so results are directly comparable.

    Creates a fresh ChatOpenAI client (make_llm) and fresh DSIExplorer(s) for each
    model via run_level -> run_benchmark, so no state leaks between models. If a
    model fails to even construct (auth error, unknown/unsupported model name,
    etc.) before any task can run, it is recorded as a failed TaskResult for every
    task in `tasks` rather than aborting the whole sweep — individual per-task
    errors are already handled inside run_benchmark. Saves one results JSON per
    model plus a combined JSON with everything.
    """
    models = models or MODELS_TO_RUN
    all_model_results: Dict[str, List[TaskResult]] = {}

    for model_name in models:
        try:
            model_llm = make_llm(model_name)
            results = run_level(level_label, tasks, llm=model_llm, model_label=model_name)
        except Exception as e:
            print(f"[ERROR] Model `{model_name}` failed entirely: {e!r}\n")
            results = [
                TaskResult(
                    task=t.query, level=t.level, category=t.category, query=t.query,
                    response="", tool_calls=[], elapsed_sec=0.0, tokens=0,
                    passed=False, score=0.0, notes=f"model_error: {e!r}", model=model_name,
                )
                for t in tasks
            ]
        all_model_results[model_name] = results

    combined = [r for model_results in all_model_results.values() for r in model_results]
    save_results(combined, f"{level_label}__all_models")
    return all_model_results


def run_prompt_variant_sweep(
    base_tasks: List[BenchmarkTask],
    models: Optional[List[str]] = None,
) -> Dict[str, List[TaskResult]]:
    """Run every (task, prompt_style) combination from expand_prompt_variants(base_tasks)
    once per model, to test RQ: does prompt phrasing/quality affect success rate, holding
    the underlying task and ground truth fixed?

    Unlike run_level (which only isolates Level 1 tasks), every run here gets a FRESH
    DSIExplorer regardless of level -- isolated=True always -- so a later-run style variant
    of a task never benefits from conversational state (e.g. an already-loaded DB) left over
    by an earlier-run style variant of the SAME task. That strict isolation is what makes
    this an apples-to-apples test of phrasing alone rather than turn-order leakage.

    Scoring is unchanged: each expanded copy still routes through score_task exactly like
    its base task (same expected_tools/expected_count/response_keywords/success_criteria/
    etc.), since only query/steps differ between styles.
    """
    models = models or MODELS_TO_RUN
    variant_tasks = expand_prompt_variants(base_tasks)
    n_styles = len(variant_tasks) // len(base_tasks) if base_tasks else 0
    display(Markdown(
        f"## Running prompt-variant sweep — {len(base_tasks)} base tasks × {n_styles} styles "
        f"= {len(variant_tasks)} runs per model"
    ))
    all_model_results: Dict[str, List[TaskResult]] = {}

    for model_name in models:
        try:
            model_llm = make_llm(model_name)
            results = run_benchmark(variant_tasks, isolated=True, llm=model_llm, model_label=model_name)
        except Exception as e:
            print(f"[ERROR] Model `{model_name}` failed entirely: {e!r}\n")
            results = [
                TaskResult(
                    task_id=t.task_id, level=t.level, category=t.category, query=t.query,
                    response="", tool_calls=[], elapsed_sec=0.0, tokens=0,
                    passed=False, score=0.0, notes=f"model_error: {e!r}", model=model_name,
                )
                for t in variant_tasks
            ]
        all_model_results[model_name] = results

    combined = [r for model_results in all_model_results.values() for r in model_results]
    save_results(combined, "prompt_variant_sweep__all_models")
    return all_model_results

### Level 1 — Atomic

In [13]:
# # Runs LEVEL1_TASKS once per model in MODELS_TO_RUN (defaults to POSSIBLE_MODELS),
# # each with its own fresh ChatOpenAI client + DSIExplorer(s), so results are comparable.
# # Set LEVELS_SKIP_NETWORK=1 beforehand to skip requires_network tasks for every model.
# l1_model_results = run_benchmark_all_models(LEVEL1_TASKS, MODELS_TO_RUN, level_label="level1_atomic")

# # Flatten into one dataframe, tagged by model, for side-by-side comparison
# l1_results = [r for results in l1_model_results.values() for r in results]
# l1_df = results_to_dataframe(l1_results, LEVEL1_TASKS)
# display(l1_df)

### Model Comparison

Pass rate, score, latency, and token usage per model — use this to compare `POSSIBLE_MODELS` head-to-head on the same task set.

In [14]:
# display(Markdown("### Overall summary by model"))
# if "judge_efficiency_score" not in l1_df.columns:
#     l1_df["judge_efficiency_score"] = float("nan")  # older results predating score_efficiency
# model_summary = (
#     l1_df.groupby("model")
#     .agg(
#         pass_rate=("passed", "mean"),
#         avg_score=("score", "mean"),
#         avg_efficiency=("judge_efficiency_score", "mean"),
#         avg_tokens=("tokens", "mean"),
#         avg_elapsed_sec=("elapsed_sec", "mean"),
#         n_tasks=("task_id", "count"),
#     )
#     .round(2)
#     .sort_values("pass_rate", ascending=False)
# )
# display(model_summary)

# display(Markdown("### Pass rate by model and category"))
# category_pivot = l1_df.pivot_table(index="model", columns="category", values="passed", aggfunc="mean").round(2)
# display(category_pivot)

# display(Markdown("### Pass/fail matrix by task and model"))
# if "query" not in l1_df.columns:
#     # l1_df was built before results_to_dataframe() added the `query` column -- backfill
#     # it here instead of forcing a full (slow, LLM-calling) re-run of the benchmark cell.
#     _task_query_by_id = {t.task_id: t.query for t in LEVEL1_TASKS}
#     l1_df["query"] = l1_df["task_id"].map(_task_query_by_id).fillna(l1_df["task_id"])
# task_pivot = l1_df.pivot_table(index="query", columns="model", values="passed", aggfunc="first")
# task_pivot.index = [textwrap.fill(q, width=60).replace("\n", "<br>") for q in task_pivot.index]
# display(HTML(task_pivot.to_html(escape=False)))

### Level 2 — Workflow

In [15]:
LEVEL2_TASKS = [
    # query catalog -> download file -> query a second DB (2 tools, cross-file)
    BenchmarkTask(
        task_id="L2_search_download_summarize_standard",
        level=2, category="Analysis & Summarization",
        query=(
            "Find the NIF simulation dataset in the Ocean 11 catalog, download the file for "
            "sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset "
            "and how many discrete timesteps the simulation has."
        ),
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["Sauppe", "20"],
        requires_network=True,
        success_criteria=(
            "Should query the Ocean 11 catalog to find the NIF dataset's DB link, download the "
            "specified simulation file, and query nif.db to report both the owner's name "
            "(Joshua Paul Sauppe) and the timestep count (20) without redundant re-exploration."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),
        
    # query catalog -> download file -> query a second DB (2 tools, cross-file)
    BenchmarkTask(
        task_id="L2_search_download_summarize_novice",
        level=2, category="Analysis & Summarization",
        query=(
            "Can you find the NIF dataset in the Ocean 11 catalog, grab the file for the simulation named nifcylxyz_hdf5_plt_cnt_0000.flash, and then tell me who owns it and how many timesteps it has?"
        ),
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["Sauppe", "20"],
        requires_network=True,
        success_criteria=(
            "Should query the Ocean 11 catalog to find the NIF dataset's DB link, download the "
            "specified simulation file, and query nif.db to report both the owner's name "
            "(Joshua Paul Sauppe) and the timestep count (20) without redundant re-exploration."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),

    # query catalog -> download file -> query a second DB (2 tools, cross-file)
    BenchmarkTask(
        task_id="L2_search_download_summarize_expert",
        level=2, category="Analysis & Summarization",
        query=(
            "Resolve the NIF dataset record in the Ocean 11 catalog, retrieve the artifact for sim_name=nifcylxyz_hdf5_plt_cnt_0000.flash via download_file_tool, then report the dataset's point-of-contact and the distinct timestep count."
        ),
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["Sauppe", "20"],
        requires_network=True,
        success_criteria=(
            "Should query the Ocean 11 catalog to find the NIF dataset's DB link, download the "
            "specified simulation file, and query nif.db to report both the owner's name "
            "(Joshua Paul Sauppe) and the timestep count (20) without redundant re-exploration."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),

    #cross-referencing a static doc file against live schema, not just querying one source
    BenchmarkTask(
        task_id="L2_poisson_glossary_audit_1",
        level=2, category="Data Preparation",
        query=(
            "The Poisson dataset has a field glossary (poisson_description.yaml) describing its columns "
            "as: name, LAUR, example, environment, data, statistics. Load poisson.db, compare its ACTUAL "
            "table schema against that documented list of fields, and report any column that exists in "
            "the real data but is NOT documented in the glossary."
        ),

        expected_tools=["load_dsi_tool", "python_repl_tool"],
        response_keywords=["argements"],
        success_criteria=(
            "Should inspect poisson.db's real schema (name, LAUR, example, environment, "
            "argements, data, statistics), compare it field-by-field against the glossary's "
            "documented fields (name, LAUR, example, environment, data, statistics), and "
            "correctly identify 'argements' as the one column present in the data but "
            "undocumented in the glossary -- without inventing a different missing field or "
            "claiming the schema and glossary fully match."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            "schema",
            "read data/oceans_11/poisson_description.yaml",
            "diff schema fields vs glossary fields",
        ],
    ),

    #cross-referencing a static doc file against live schema, not just querying one source
    BenchmarkTask(
        task_id="L2_poisson_glossary_audit_2",
        level=2, category="Data Preparation",
        query=(
            "The Poisson dataset has a field glossary file at data/oceans_11/poisson_description.yaml describing its documented columns. Load poisson.db, then read the glossary file and compare the ACTUAL table schema against the fields documented in the glossary. Report any column that exists in the real data but is NOT documented in the glossary."
        ),

        expected_tools=["load_dsi_tool", "python_repl_tool"],
        response_keywords=["argements"],
        success_criteria=(
            "Should inspect poisson.db's real schema (name, LAUR, example, environment, "
            "argements, data, statistics), compare it field-by-field against the glossary's "
            "documented fields (name, LAUR, example, environment, data, statistics), and "
            "correctly identify 'argements' as the one column present in the data but "
            "undocumented in the glossary -- without inventing a different missing field or "
            "claiming the schema and glossary fully match."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            "schema",
            "read data/oceans_11/poisson_description.yaml",
            "diff schema fields vs glossary fields",
        ],
    ),

    # requires genuine aggregation/extremum-finding across many rows
    BenchmarkTask(
        task_id="L2_nif_dens_max_extremum_standard",
        level=2, category="Analysis & Summarization",
        query=(
            "Across all 20 timesteps of the NIF simulation, which timestep has the highest "
            "maximum density (dens_max), and what is that value?"
        ),
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["6", "7.9"],
        success_criteria=(
            "Should aggregate/compare dens_max across ALL 20 nif_metadata rows (e.g. via "
            "MAX()/ORDER BY, or by scanning and comparing every row) rather than reporting a "
            "single arbitrary row, and correctly identify timestep 6 with a dens_max of "
            "approximately 7.95 as the highest."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, dens_max FROM nif_metadata ORDER BY dens_max DESC LIMIT 1"',
        ],
    ),

    # requires genuine aggregation/extremum-finding across many rows
    BenchmarkTask(
        task_id="L2_nif_dens_max_extremum_expert",
        level=2, category="Analysis & Summarization",
        query=(
            "Across all nif_metadata rows, identify the timestep with the maximum value of dens_max and report that value."
        ),
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["6", "7.9"],
        success_criteria=(
            "Should aggregate/compare dens_max across ALL 20 nif_metadata rows (e.g. via "
            "MAX()/ORDER BY, or by scanning and comparing every row) rather than reporting a "
            "single arbitrary row, and correctly identify timestep 6 with a dens_max of "
            "approximately 7.95 as the highest."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, dens_max FROM nif_metadata ORDER BY dens_max DESC LIMIT 1"',
        ],
    ),

    #tests whether the agent fabricates a confident-but-wrong distinction when the data actually shows a tie
    BenchmarkTask(
        task_id="L2_nif_num_arrays_tie_standard",
        level=2, category="Analysis & Summarization",
        query=(
            "Which timestep in the NIF simulation has the most data arrays recorded (num_arrays), "
            "and by how many more than the next-highest timestep?"
        ),
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["9"],
        success_criteria=(
            "Should check num_arrays across ALL 20 nif_metadata rows and recognize they are "
            "ALL tied at exactly 9 -- there is no single highest timestep, so the correct "
            "answer explicitly states they're tied (difference of 0), rather than naming an "
            "arbitrary single timestep as 'the winner' or fabricating a nonzero difference."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT DISTINCT num_arrays FROM nif_metadata"',
        ],
    ),

    #tests whether the agent fabricates a confident-but-wrong distinction when the data actually shows a tie
    BenchmarkTask(
        task_id="L2_nif_num_arrays_tie_terse",
        level=2, category="Analysis & Summarization",
        query=(
            "max num_arrays timestep + margin over 2nd?"
        ),
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        response_keywords=["9"],
        success_criteria=(
            "Should check num_arrays across ALL 20 nif_metadata rows and recognize they are "
            "ALL tied at exactly 9 -- there is no single highest timestep, so the correct "
            "answer explicitly states they're tied (difference of 0), rather than naming an "
            "arbitrary single timestep as 'the winner' or fabricating a nonzero difference."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT DISTINCT num_arrays FROM nif_metadata"',
        ],
    ),
]

In [16]:
l2_model_results = run_benchmark_all_models(LEVEL2_TASKS, MODELS_TO_RUN, level_label="level2_workflow")
l2_results = [r for results in l2_model_results.values() for r in results]
l2_df = results_to_dataframe(l2_results, LEVEL2_TASKS)
display(l2_df)

## Running level2_workflow — model: `aws-gov.gpt-5.4` (9 tasks)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_10_15__07_49


### `L2_search_download_summarize_standard` — Level 2 / Analysis & Summarization

**Query:** Find the NIF simulation dataset in the Ocean 11 catalog, download the file for sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset and how many discrete timesteps the simulation has.

I found the dataset and loaded its DSI file.

- Dataset: 3D FLASH Computation of National Ignition Facility Shot
- Owner/contact: Joshua Paul Sauppe
- Email: jpsauppe@lanl.gov
- Organization: DOE:LANL

For sim_name `nifcylxyz_hdf5_plt_cnt_0000.flash`:
- Discrete timesteps: 61
- Timestep range: 0 to 60

Download status:
- I attempted to download `nifcylxyz_hdf5_plt_cnt_0000.flash`, but the download failed because the remote host `gsdview.llnl.gov` could not be resolved from this environment.

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'query_dsi_tool', 'query_dsi_tool'] | 19.4s | 6214 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] aws-gov.gpt-5.4 | L2_search_download_summarize_standard — score=0.755
Wall-clock total: 28.61s
  step:1                             19.41s  ( 67.9%)  x1 calls, avg 19.41s
  score:efficiency                    4.70s  ( 16.4%)  x1 calls, avg 4.70s
  score:level2                        4.50s  ( 15.7%)  x1 calls, avg 4.50s
  score:checkpoint_opportunities      0.00s  (  0.0%)  x1 calls, avg 0.00s
  score:classify_failure              0.00s  (  0.0%)  x1 calls, avg 0.00s

...

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'download_file_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'load_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'python_repl_tool', 'python_repl_tool', 'python_repl_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool'] | 10.2s | 20064 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_nif_num_arrays_tie_standard — score=0.87
Wall-clock total: 19.81s
  step:1                             10.24s  ( 51.7%)  x1 calls, avg 10.24s
  score:efficiency                    4.98s  ( 25.1%)  x1 calls, avg 4.98s
  

### `L2_nif_num_arrays_tie_terse` — Level 2 / Analysis & Summarization

**Query:** max num_arrays timestep + margin over 2nd?

## Results

**Max num_arrays:** 9 (all timesteps)

**Margin over 2nd:** 0

All 20 timesteps have identical num_arrays = 9, so there's no difference between the highest and second-highest.

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'download_file_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'load_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'python_repl_tool', 'python_repl_tool', 'python_repl_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool'] | 5.7s | 20467 tokens
DSI API calls: ['__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_nif_num_arrays_tie_terse — score=0.893
Wall-clock total: 16.07s
  step:1                              5.71s  ( 35.5%)  x1 calls, avg 5.71s
  score:efficiency                    5.18s  ( 32.2%)  x1 calls, avg 5.18s
  score:level2    

,model,task_id,query,prompt_style,base_task_id,level,category,passed,score,elapsed_sec,...,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_process_quality_score,judge_process_quality_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L2_search_download_summarize_standard,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize_standard,2,Analysis & Summarization,True,0.755,19.4,...,False,False,1,1,0,None,0.510,[aws-gov.gpt-5.4] The agent mostly followed th...,0.270,[aws-gov.gpt-5.4] The agent took a much more w...
1,aws-gov.gpt-5.4,L2_search_download_summarize_novice,Can you find the NIF dataset in the Ocean 11 c...,standard,L2_search_download_summarize_novice,2,Analysis & Summarization,True,0.715,1.5,...,False,False,1,1,0,None,0.430,[aws-gov.gpt-5.4] The agent did locate the NIF...,0.340,[aws-gov.gpt-5.4] The agent took 10 tool calls...
2,aws-gov.gpt-5.4,L2_search_download_summarize_expert,Resolve the NIF dataset record in the Ocean 11...,standard,L2_search_download_summarize_expert,2,Analysis & Summarization,True,0.670,9.3,...,False,False,2,2,0,None,0.340,[aws-gov.gpt-5.4] The agent did locate the NIF...,0.205,[aws-gov.gpt-5.4] The agent was far less effic...
3,aws-gov.gpt-5.4,L2_poisson_glossary_audit_1,The Poisson dataset has a field glossary (pois...,standard,L2_poisson_glossary_audit_1,2,Data Preparation,False,0.122,2.6,...,False,False,2,2,0,UNKNOWN,0.245,[aws-gov.gpt-5.4] Although the final answer na...,0.140,[aws-gov.gpt-5.4] The agent eventually loaded ...
4,aws-gov.gpt-5.4,L2_poisson_glossary_audit_2,The Poisson dataset has a field glossary file ...,standard,L2_poisson_glossary_audit_2,2,Data Preparation,True,0.818,12.4,...,False,False,2,2,0,None,0.635,[aws-gov.gpt-5.4] The agent eventually produce...,0.115,[aws-gov.gpt-5.4] The agent eventually inspect...
5,aws-gov.gpt-5.4,L2_nif_dens_max_extremum_standard,"Across all 20 timesteps of the NIF simulation,...",standard,L2_nif_dens_max_extremum_standard,2,Analysis & Summarization,True,0.965,5.1,...,False,False,2,2,0,None,0.930,[aws-gov.gpt-5.4] Despite much irrelevant expl...,0.115,[aws-gov.gpt-5.4] The agent eventually issued ...
6,aws-gov.gpt-5.4,L2_nif_dens_max_extremum_expert,"Across all nif_metadata rows, identify the tim...",standard,L2_nif_dens_max_extremum_expert,2,Analysis & Summarization,True,0.912,2.9,...,False,False,2,2,0,None,0.825,[aws-gov.gpt-5.4] Although the agent eventuall...,0.090,[aws-gov.gpt-5.4] Although the underlying DSI ...
7,aws-gov.gpt-5.4,L2_nif_num_arrays_tie_standard,Which timestep in the NIF simulation has the m...,standard,L2_nif_num_arrays_tie_standard,2,Analysis & Summarization,True,0.660,4.3,...,False,False,2,2,0,None,0.320,[aws-gov.gpt-5.4] The agent eventually noted a...,0.115,[aws-gov.gpt-5.4] Although the final DSI inter...
8,aws-gov.gpt-5.4,L2_nif_num_arrays_tie_terse,max num_arrays timestep + margin over 2nd?,standard,L2_nif_num_arrays_tie_terse,2,Analysis & Summarization,True,0.718,3.1,...,False,False,2,2,0,None,0.435,[aws-gov.gpt-5.4] The agent eventually recogni...,0.200,[aws-gov.gpt-5.4] The reference required just ...
9,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_search_download_summarize_standard,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize_standard,2,Analysis & Summarization,True,0.830,38.9,...,False,False,2,2,0,None,0.660,[aws-gov.gpt-5.4] The agent ultimately found n...,0.295,[aws-gov.gpt-5.4] The agent reached the needed...


In [17]:
display(Markdown("### Overall summary by model"))
if "judge_efficiency_score" not in l2_df.columns:
    l2_df["judge_efficiency_score"] = float("nan")  # older results predating score_efficiency
model_summary = (
    l2_df.groupby("model")
    .agg(
        pass_rate=("passed", "mean"),
        avg_score=("score", "mean"),
        avg_efficiency=("judge_efficiency_score", "mean"),
        avg_tokens=("tokens", "mean"),
        avg_elapsed_sec=("elapsed_sec", "mean"),
        n_tasks=("task_id", "count"),
    )
    .round(2)
    .sort_values("pass_rate", ascending=False)
)
display(model_summary)

display(Markdown("### Pass rate by model and category"))
category_pivot = l2_df.pivot_table(index="model", columns="category", values="passed", aggfunc="mean").round(2)
display(category_pivot)

display(Markdown("### Pass/fail matrix by task and model"))
if "query" not in l2_df.columns:
    # l1_df was built before results_to_dataframe() added the `query` column -- backfill
    # it here instead of forcing a full (slow, LLM-calling) re-run of the benchmark cell.
    _task_query_by_id = {t.task_id: t.query for t in LEVEL1_TASKS}
    l2_df["query"] = l2_df["task_id"].map(_task_query_by_id).fillna(l2_df["task_id"])
task_pivot = l2_df.pivot_table(index="query", columns="model", values="passed", aggfunc="first")
task_pivot.index = [textwrap.fill(q, width=60).replace("\n", "<br>") for q in task_pivot.index]
display(HTML(task_pivot.to_html(escape=False)))

display(Markdown("### Where does the time go? (aggregate timing breakdown)"))
l2_timing_df = timing_breakdown_dataframe(l2_results)
if l2_timing_df.empty:
    print("(no timing data -- re-run the benchmark cell above to populate timing_log)")
else:
    # Top-level (non-nested) phases only -- these are sequential/non-overlapping per task, so
    # summing them gives a real wall-clock total. llm_call/tool:* are excluded here because
    # they're recorded *inside* a step:*/setup_query phase and summing both double-counts.
    top_level_df = l2_timing_df[~l2_timing_df["is_nested"]]
    display(Markdown("#### Top-level phases (sums to real wall-clock time; no double-counting)"))
    display(
        top_level_df.groupby("phase_group")["elapsed_sec"]
        .agg(total_sec="sum", mean_sec="mean", count="count")
        .round(3)
        .sort_values("total_sec", ascending=False)
    )
    display(Markdown("#### By model x top-level phase group (total seconds)"))
    display(
        top_level_df.groupby(["model", "phase_group"])["elapsed_sec"]
        .sum()
        .unstack("phase_group")
        .round(2)
    )

    # Nested breakdown (llm_call/tool:*) -- shown separately since it's already included in
    # the step:*/setup_query totals above, not additional time on top of them.
    nested_df = l2_timing_df[l2_timing_df["is_nested"]]
    if not nested_df.empty:
        display(Markdown("#### Nested within step:*/setup_query (NOT additional time -- already counted above)"))
        display(
            nested_df.groupby("phase_group")["elapsed_sec"]
            .agg(total_sec="sum", mean_sec="mean", count="count")
            .round(3)
            .sort_values("total_sec", ascending=False)
        )

### Overall summary by model

,pass_rate,avg_score,avg_efficiency,avg_tokens,avg_elapsed_sec,n_tasks
model,,,,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,1.00,0.84,0.21,16224.44,14.78,9
aws-gov.gpt-5.4,0.89,0.70,0.18,9856.33,6.73,9


### Pass rate by model and category

category,Analysis & Summarization,Data Preparation
model,,
anthropic.claude-sonnet-4-5-20250929-v1:0,1.0,1.0
aws-gov.gpt-5.4,1.0,0.5


### Pass/fail matrix by task and model

...

,total_sec,mean_sec,count
phase_group,,,
step,193.616,10.756,18
score,166.295,2.310,72


#### By model x top-level phase group (total seconds)

phase_group,score,step
model,,
anthropic.claude-sonnet-4-5-20250929-v1:0,82.01,133.13
aws-gov.gpt-5.4,84.28,60.49


#### Nested within step:*/setup_query (NOT additional time -- already counted above)

,total_sec,mean_sec,count
phase_group,,,
llm_call,189.595,2.597,73
tool,3.756,0.063,60


### Level 3 — Discovery

In [18]:
# l3_model_results = run_benchmark_all_models(LEVEL3_TASKS, MODELS_TO_RUN, level_label="level3_discovery")
# l3_results = [r for results in l3_model_results.values() for r in results]
# l3_df = results_to_dataframe(l3_results, LEVEL3_TASKS)
# display(l3_df)

### Calibration — Does the Agent Know What It Doesn't Know? (RQ4)

In [19]:
# cal_model_results = run_benchmark_all_models(CALIBRATION_TASKS, MODELS_TO_RUN, level_label="calibration")
# cal_results = [r for results in cal_model_results.values() for r in results]
# cal_df = results_to_dataframe(cal_results, CALIBRATION_TASKS)
# display(cal_df)

# display(Markdown("### Calibration precision by model (share of trap tasks correctly abstained on)"))
# display(cal_df.groupby("model")["passed"].mean().round(2).rename("abstention_accuracy").to_frame())

# display(Markdown("### Hallucinated (overconfident) responses"))
# display(cal_df[cal_df["hallucination_detected"]][["model", "task_id", "response"]])

### Prompt-Variant Sweep — Does Prompt Quality/Style Affect Success Rate?

Every Level 1 and Level 2 task above also has 5 hand-authored alternate phrasings
(`task.prompt_variants`) of the *exact same underlying ask* — the ground truth, expected
tools, and scoring rubric are identical across styles; only the wording/register changes:

- **novice** — plain-language, non-technical phrasing a first-time user might type
- **expert** — technical/jargon-heavy phrasing using DSI field names and operation terms
- **terse** — minimal, keyword-only phrasing
- **verbose** — over-explained, context-heavy phrasing with extra framing
- **casual** — informal, lowercase, abbreviation-heavy phrasing

`expand_prompt_variants()` turns each base task into 6 runs (`standard` + the 5 styles
above), and `run_prompt_variant_sweep()` executes every one of those **in full isolation**
(a fresh `DSIExplorer` per run, regardless of level) so a later-run style never benefits
from conversational state left over by an earlier-run style of the *same* task — otherwise
a model could "pass" a harder phrasing for free just because an earlier easier phrasing
already loaded the right database in a shared session.

**Cost/time note:** this expands 13 Level 1 + 11 Level 2 = 24 base tasks × 6 phrasings ×
`len(MODELS_TO_RUN)` models into **144 × n_models agent runs** — substantially more calls
than any other section in this notebook. Consider trying it on a small subset first (e.g.
`PROMPT_VARIANT_BASE_TASKS[:3]`) before running the full sweep, and/or temporarily reducing
`MODELS_TO_RUN` to one model.

In [20]:
PROMPT_VARIANT_BASE_TASKS = list(LEVEL2_TASKS)
_pv_preview = expand_prompt_variants(PROMPT_VARIANT_BASE_TASKS)
_n_styles = len(_pv_preview) // len(PROMPT_VARIANT_BASE_TASKS) if PROMPT_VARIANT_BASE_TASKS else 0
print(
    f"{len(PROMPT_VARIANT_BASE_TASKS)} base tasks (Level 2) × {_n_styles} styles "
    f"(standard + 5 variants) = {len(_pv_preview)} runs per model"
)

9 base tasks (Level 2) × 1 styles (standard + 5 variants) = 9 runs per model


In [21]:
pv_model_results = run_prompt_variant_sweep(PROMPT_VARIANT_BASE_TASKS, MODELS_TO_RUN)
pv_results = [r for results in pv_model_results.values() for r in results]
pv_df = results_to_dataframe(pv_results, expand_prompt_variants(PROMPT_VARIANT_BASE_TASKS))
display(pv_df)

## Running prompt-variant sweep — 9 base tasks × 1 styles = 9 runs per model

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_10_15__07_55


### `L2_search_download_summarize_standard` — Level 2 / Analysis & Summarization

**Query:** Find the NIF simulation dataset in the Ocean 11 catalog, download the file for sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset and how many discrete timesteps the simulation has.

Closing this instance of DSI()


Downloaded:
`/Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_10_15__07_55/nifcylxyz_hdf5_plt_cnt_0000.flash`

Dataset owner:
Joshua Paul Sauppe (`jpsauppe@lanl.gov`), originating organization `DOE:LANL`

Discrete timesteps for simulation/file `nifcylxyz_hdf5_plt_cnt_0000`:
`1` timestep (`timestep = 0`)

...

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool'] | 28.1s | 6631 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_nif_num_arrays_tie_standard — score=0.985
Wall-clock total: 36.56s
  step:1                             28.08s  ( 76.8%)  x1 calls, avg 28.08s
  score:level2                        4.47s  ( 12.2%)  x1 calls, avg 4.47s
  score:efficiency                    4.01s  ( 11.0%)  x1 calls, avg 4.01s
  score:checkpoint_opportunities      0.00s  (  0.0%)  x1 calls, avg 0.00s
  score:classify_failure              0.00s  (  0.0%)  x1 calls, avg 0.00s
  -- nested within the phases above (llm/tool calls inside each step/setup_query; not additional wall-clock time) --
    within step:1 (28.08s):
      llm

### `L2_nif_num_arrays_tie_terse` — Level 2 / Analysis & Summarization

**Query:** max num_arrays timestep + margin over 2nd?

I need more context about which dataset you're referring to. Could you clarify which dataset contains the "num_arrays" field you're asking about? 

Based on the available datasets with DSI databases:
1. **3D FLASH Computation of National Ignition Facility Shot** (nif.db)
2. **Poisson's Equations for Electrostatics** (poisson.db)

Which dataset would you like me to analyze, or should I check both?

Tools called: ['query_dsi_tool', 'query_dsi_tool'] | 9.2s | 5559 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close']
[FAIL] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_nif_num_arrays_tie_terse — score=0.02
Wall-clock total: 21.00s
  step:1                              9.24s  ( 44.0%)  x1 calls, avg 9.24s
  score:classify_failure              4.34s  ( 20.7%)  x1 calls, avg 4.34s
  score:level2                        3.88s  ( 18.5%)  x1 calls, avg 3.88s
  score:efficiency                    3.54s  ( 16.9%)  x1 calls, avg 3.54s
  score:checkpoint_opportunities      0.00s  (  0.0%)  x1 calls, avg 0.00s
  -- nested within the phases above (llm/tool calls inside each step/setup_query; not additional wall-clock time) --
    within step:1 (9.24s):
      llm_call                        9.22s  ( 99.8% of step:1)  x3 calls, avg 3.07s
      tool:query_dsi_tool             0.01s  (  0.1% of step:1)  x2 calls, avg 0.00s

Saved: /Users/hhn/lanl/dsi/tools/ai_search/le

,model,task_id,query,prompt_style,base_task_id,level,category,passed,score,elapsed_sec,...,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_process_quality_score,judge_process_quality_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L2_search_download_summarize_standard,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize_standard,2,Analysis & Summarization,True,0.730,8.3,...,False,False,1,1,0,None,0.460,[aws-gov.gpt-5.4] The agent sensibly found and...,0.510,[aws-gov.gpt-5.4] The agent completed the task...
1,aws-gov.gpt-5.4,L2_search_download_summarize_novice,Can you find the NIF dataset in the Ocean 11 c...,standard,L2_search_download_summarize_novice,2,Analysis & Summarization,False,0.287,7.5,...,False,False,0,0,0,UNKNOWN,0.575,[aws-gov.gpt-5.4] The workflow correctly found...,0.510,[aws-gov.gpt-5.4] The agent found the right da...
2,aws-gov.gpt-5.4,L2_search_download_summarize_expert,Resolve the NIF dataset record in the Ocean 11...,standard,L2_search_download_summarize_expert,2,Analysis & Summarization,True,0.932,6.8,...,False,False,1,1,0,None,0.865,[aws-gov.gpt-5.4] The workflow mostly followed...,0.520,[aws-gov.gpt-5.4] The agent completed the task...
3,aws-gov.gpt-5.4,L2_poisson_glossary_audit_1,The Poisson dataset has a field glossary (pois...,standard,L2_poisson_glossary_audit_1,2,Data Preparation,False,0.430,5.6,...,False,False,0,0,0,PROMPT_GAP,0.860,[aws-gov.gpt-5.4] The workflow loaded the targ...,0.290,[aws-gov.gpt-5.4] The agent spent multiple unn...
4,aws-gov.gpt-5.4,L2_poisson_glossary_audit_2,The Poisson dataset has a field glossary file ...,standard,L2_poisson_glossary_audit_2,2,Data Preparation,True,0.905,5.6,...,False,False,0,0,0,None,0.810,[aws-gov.gpt-5.4] The agent read the glossary ...,0.510,[aws-gov.gpt-5.4] The agent completed the esse...
5,aws-gov.gpt-5.4,L2_nif_dens_max_extremum_standard,"Across all 20 timesteps of the NIF simulation,...",standard,L2_nif_dens_max_extremum_standard,2,Analysis & Summarization,True,0.995,3.9,...,False,False,0,0,0,None,0.990,[aws-gov.gpt-5.4] The agent correctly located ...,0.425,[aws-gov.gpt-5.4] The agent found the correct ...
6,aws-gov.gpt-5.4,L2_nif_dens_max_extremum_expert,"Across all nif_metadata rows, identify the tim...",standard,L2_nif_dens_max_extremum_expert,2,Analysis & Summarization,True,0.995,4.8,...,False,False,0,0,0,None,0.990,[aws-gov.gpt-5.4] The agent correctly located ...,0.360,[aws-gov.gpt-5.4] The agent found the correct ...
7,aws-gov.gpt-5.4,L2_nif_num_arrays_tie_standard,Which timestep in the NIF simulation has the m...,standard,L2_nif_num_arrays_tie_standard,2,Analysis & Summarization,True,0.992,7.7,...,False,False,0,0,0,None,0.985,[aws-gov.gpt-5.4] The agent used sensible SQL ...,0.190,[aws-gov.gpt-5.4] The agent took a notably was...
8,aws-gov.gpt-5.4,L2_nif_num_arrays_tie_terse,max num_arrays timestep + margin over 2nd?,standard,L2_nif_num_arrays_tie_terse,2,Analysis & Summarization,True,0.688,7.7,...,False,False,0,0,0,None,0.375,[aws-gov.gpt-5.4] The agent did inspect the re...,0.190,[aws-gov.gpt-5.4] The agent eventually hit the...
9,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_search_download_summarize_standard,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize_standard,2,Analysis & Summarization,True,0.830,38.3,...,False,False,2,2,0,None,0.660,[aws-gov.gpt-5.4] The agent ultimately found t...,0.315,[aws-gov.gpt-5.4] The agent reached the goal b...


In [22]:
# display(Markdown("### Pass rate / avg score by prompt style (across all tasks and models)"))
# style_summary = (
#     pv_df.groupby("prompt_style")
#     .agg(pass_rate=("passed", "mean"), avg_score=("score", "mean"), n=("task_id", "count"))
#     .round(3)
#     .reindex(["standard", "novice", "expert", "terse", "verbose", "casual"])
# )
# display(style_summary)

# display(Markdown("### Pass rate by base task and prompt style (1.0 = always passed at that style)"))
# style_pivot = pv_df.pivot_table(index="base_task_id", columns="prompt_style", values="passed", aggfunc="mean")
# style_pivot = style_pivot[[c for c in ["standard", "novice", "expert", "terse", "verbose", "casual"] if c in style_pivot.columns]]
# display(style_pivot)

# display(Markdown(
#     "### Per-task phrasing sensitivity (spread in pass rate across styles, per model)\n\n"
#     "0 = fully robust to phrasing (same outcome regardless of wording); higher = more "
#     "sensitive to HOW the task was asked, holding the task itself constant."
# ))
# sensitivity = (
#     pv_df.groupby(["base_task_id", "model", "prompt_style"])["passed"].mean()
#     .reset_index()
#     .groupby(["base_task_id", "model"])["passed"]
#     .agg(lambda s: s.max() - s.min())
#     .reset_index(name="pass_rate_spread")
#     .sort_values("pass_rate_spread", ascending=False)
# )
# display(sensitivity)

# display(Markdown("### Overall: is either model more sensitive to prompt phrasing than the other?"))
# model_sensitivity = sensitivity.groupby("model")["pass_rate_spread"].mean().round(3)
# display(model_sensitivity)

## All Results

In [23]:
# all_results = l1_results + l2_results + l3_results + cal_results
# all_tasks = ALL_TASKS
# summary_df = results_to_dataframe(all_results, all_tasks)

# # --- RQ2: how does performance change from Level 1 (atomic) -> 2 (workflow) -> 3 (discovery)? ---
# display(Markdown("### RQ2 — Pass rate / score / efficiency / latency by complexity level and model"))
# level_pivot = (
#     summary_df[summary_df["level"].isin([1, 2, 3])]
#     .groupby(["model", "level"])
#     .agg(
#         pass_rate=("passed", "mean"),
#         avg_score=("score", "mean"),
#         avg_efficiency=("judge_efficiency_score", "mean") if "judge_efficiency_score" in summary_df.columns else ("score", "mean"),
#         avg_elapsed_sec=("elapsed_sec", "mean"),
#         avg_tokens=("tokens", "mean"),
#         n_tasks=("task_id", "count"),
#     )
#     .round(2)
# )
# display(level_pivot)

# try:
#     import matplotlib.pyplot as plt

#     fig, ax = plt.subplots(figsize=(6, 4))
#     for model_name, g in summary_df[summary_df["level"].isin([1, 2, 3])].groupby("model"):
#         by_level = g.groupby("level")["passed"].mean()
#         ax.plot(by_level.index, by_level.values, marker="o", label=model_name)
#     ax.set_xlabel("Complexity level")
#     ax.set_ylabel("Pass rate")
#     ax.set_xticks([1, 2, 3])
#     ax.set_title("Pass rate vs. task complexity (RQ2)")
#     ax.legend(fontsize=8)
#     plt.show()
# except Exception as e:
#     print(f"(skipping plot: {e!r})")

# # --- RQ4: calibration -- is the agent well-calibrated about what it doesn't know? ---
# display(Markdown("### RQ4 — Calibration: abstention accuracy on trap tasks, by model"))
# cal_only = summary_df[summary_df["expects_abstention"] == True]
# if not cal_only.empty:
#     display(
#         cal_only.groupby("model")
#         .agg(abstention_accuracy=("passed", "mean"), hallucinations=("hallucination_detected", "sum"), n_traps=("task_id", "count"))
#         .round(2)
#     )

# # --- RQ5: when the agent fails, why? Failure-attribution taxonomy by model ---
# display(Markdown("### RQ5 — Failure-category distribution by model (root-cause attribution)"))
# failed_only = summary_df[summary_df["passed"] == False]
# if not failed_only.empty:
#     display(
#         failed_only.pivot_table(index="model", columns="failure_category", values="task_id", aggfunc="count", fill_value=0)
#     )

# # --- RQ3/RQ7: where would a human-in-the-loop checkpoint have fired? (metrics only, see BENCHMARK_METHODOLOGY.md) ---
# display(Markdown("### RQ3/RQ7 — Checkpoint-opportunity counts vs. outcome, by model"))
# display(
#     summary_df.groupby("model")
#     .agg(
#         avg_checkpoint_opportunities=("checkpoint_opportunity_count", "mean"),
#         avg_high_stakes_actions=("high_stakes_action_count", "mean"),
#         avg_hedge_language=("hedge_language_count", "mean"),
#         pass_rate=("passed", "mean"),
#     )
#     .round(2)
# )

# # --- RQ1: does agentic assistance improve productivity vs. the human DSI-CLI baseline? ---
# display(Markdown("### RQ1 — Productivity vs. human baseline (requires reference_time_sec; see BENCHMARK_METHODOLOGY.md)"))
# display(effective_productivity(summary_df))

# display(Markdown("### Full combined results (all levels + calibration)"))
# display(summary_df)

# save_results(all_results, "all_levels_summary")

## Single-Task Runner

This cell debugs individual tasks or add custom queries.

In [24]:
# def debug_task(task_id: str, model_name: Optional[str] = None):
#     """Run and score a single task by ID for interactive debugging.

#     Looks up the task in ALL_TASKS, creates a fresh explorer, and prints pass/score.
#     Pass model_name (any entry from POSSIBLE_MODELS) to debug that task against a
#     specific model instead of the global default LLM_MODEL.
#     """
#     task = next(t for t in ALL_TASKS if t.task_id == task_id)
#     explorer = make_explorer(llm=make_llm(model_name) if model_name else None)
#     print(f"Agent workspace: {explorer.wrsk_path}")
#     try:
#         tr = run_task(explorer, task)
#         tr = score_task(task, tr)
#     except TaskTimeoutError as e:
#         print(f"[TIMEOUT] {e}")
#         tr = TaskResult(
#             task_id=task.task_id, level=task.level, category=task.category, query=task.query,
#             response="", tool_calls=[], elapsed_sec=task.timeout_sec or TASK_TIMEOUT_SEC, tokens=0,
#             passed=False, score=0.0, notes=f"timeout: {e}",
#         )
#     tr.model = model_name or LLM_MODEL
#     print(f"Model: {tr.model}")
#     print(f"Passed: {tr.passed} | Score: {tr.score}")
#     print(f"Tools: {tr.tool_calls}")
#     print(f"DSI API calls:\n{format_dsi_call_log(tr.dsi_calls)}")
#     if tr.notes:
#         print(f"Notes: {tr.notes}")
#     if tr.judge_scores:
#         print(json.dumps(tr.judge_scores, indent=2))
#     return tr

# # Example: debug_task("L1_load_ocean11", model_name="anthropic.claude-sonnet-4-5-20250929-v1:0")


In [25]:
# # Run task 1 and display its agent logs
# task = LEVEL1_TASKS[0]
# explorer = make_explorer()
# tr = run_task(explorer, task)
# tr = score_task(task, tr)

# print(f"Passed: {tr.passed} | Score: {tr.score}")
# print(f"DSI API calls:\n{format_dsi_call_log(tr.dsi_calls)}")
# print(Path(explorer.wrsk_path, "run.log").read_text())


## MCP Mode

To benchmark via MCP (atomic tools on `levels_mcp_server.py`):

1. In a terminal: `python levels_mcp_server.py`
2. Set `USE_MCP = True` in the config cell and re-run setup.

Inspector: `npx @modelcontextprotocol/inspector` → Streamable HTTP → `http://127.0.0.1:8000/mcp`
